# **Riffusion Fine-tuning for Music Sample Generation**
### ECS 7022P Computational Creativity – Assessed Practical
**Sound and Music Computing, Queen Mary University of London**  
*Adin Segall*  
*April, 2025*

This project fine-tunes the Riffusion model to generate high-quality audio samples and loops for music production.

## 1. Environment Setup

### 1.0 Mount Google Drive and Set Project Paths

Mount Google Drive to access stored resources required by this notebook, including dataset, model files, and seed images.

> **IMPORTANT SETUP STEP**  
> Before running the notebook, please ensure you've added the shared project folders to your own Drive so the notebook paths resolve correctly:
>
> 1. Open the following shared folders:
>    - Training data: commercial EDM sample pack (WAProd) — not redistributed.
>
> 2. For each folder, click the **"Add shortcut to Drive"** button and place it into your **`My Drive`** folder.
>
> 3. After mounting Drive using `drive.mount()`, ensure the folders appear at:
> ```
> /content/drive/MyDrive/
> ├── WAProd_Free_Anniversary_Collection_Vol7/
> ├── riffusion-hobby-0.3.0/
> │   └── seed_images/
> └── riffusion_finetuning/
> ```

Once the folders are added and paths are confirmed, the following code will set the working directory and path variables accordingly.

In [ ]:
# Mount Google Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

# Define the path to the riffusion project
RIFFUSION_DIR = '/content/drive/MyDrive/riffusion-hobby-0.3.0'
os.chdir(RIFFUSION_DIR)

# Add the project to Python path
if RIFFUSION_DIR not in sys.path:
    sys.path.append(RIFFUSION_DIR)

# Define paths for model and seed images
CHECKPOINT_DIR = os.path.join(RIFFUSION_DIR, 'checkpoints')
MODEL_ID = "riffusion/riffusion-model-v1"
SEED_IMAGES_DIR = os.path.join(RIFFUSION_DIR, 'seed_images')

print(f"Working directory set to: {os.getcwd()}")
print(f"Seed images directory: {SEED_IMAGES_DIR}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

### 1.1 Install Dependencies

Install required Python packages including PyTorch, Diffusers, OpenVINO, and Gradio. Specific version dependencies for Riffusion Model

In [ ]:
# Install required packages
!pip install -q --extra-index-url https://download.pytorch.org/whl/cpu torch torchaudio "diffusers>=0.16.1" "transformers>=4.33.0"
!pip install -q "optimum-intel" onnx "gradio>=3.34.0" "openvino>=2023.1.0"
!apt-get update && apt-get install -y ffmpeg

### 1.2 Import Dependencies

In [ ]:
# Core libraries
import io
import gc
import os
import sys
import time
import json
import random

from typing import Tuple, Dict, List, Optional, Union
from pathlib import Path
from tqdm import tqdm

# Scientific computing
import numpy as np
import pandas as pd

# Deep learning
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

# Computer vision and image processing
import matplotlib.pyplot as plt
from PIL import Image


# Audio processing
from scipy.io import wavfile
import torchaudio
import librosa
import pydub

# Diffusion models and transformers
from openvino.runtime import Core
from transformers import AutoTokenizer
from diffusers import DiffusionPipeline
from optimum.intel.openvino import OVStableDiffusionPipeline
from diffusers import StableDiffusionPipeline, DiffusionPipeline

# Utilities
from IPython.display import Audio, display, HTML
import ipywidgets as widgets
from tqdm.auto import tqdm
import traceback

## 2. Model Initialization and Loading

This section converts the Riffusion model using OpenVINO. We also check for device availability and verify the model via a test inference.

### 2.1 Set Up OpenVINO Pipeline

We use the `OVStableDiffusionPipeline` to convert the model into OpenVINO IR format. This allows for optimized inference on compatible hardware or the AUTO device selector.

In [ ]:
# Force garbage collection
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Check available devices
core = Core()
available_devices = core.available_devices + ["AUTO"]
print(f"Available devices: {available_devices}")

# Select device
DEVICE = "AUTO"

# Convert model
MODEL_DIR = Path(os.path.join(CHECKPOINT_DIR, "ov_pipeline"))

# Always recreate the model for reliability
if MODEL_DIR.exists():
    print(f"Removing existing model directory: {MODEL_DIR}")
    shutil.rmtree(MODEL_DIR)
    # Give the file system time to clean up
    time.sleep(1)

print("Creating fresh OpenVINO model conversion...")
pipe = OVStableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    export=True,
    device=DEVICE,
    compile=False,
    safety_checker=None,
    requires_safety_checker=False
)
pipe.half()  # Use half precision

# Optimize for inference
print("Optimizing model for inference...")
pipe.reshape(batch_size=1, height=512, width=512, num_images_per_prompt=1)
pipe.compile()

# Run a quick test to verify
print("Verifying model with test inference...")
_ = pipe("test pattern", num_inference_steps=2)
print("Model ready for inference")

## 3. Core Functions: Audio Generation Pipeline

This section defines the core components of the text-to-audio generation process, including spectrogram reconstruction and waveform synthesis.

In [5]:
# Global constants for consistent spectrogram processing
SAMPLE_RATE = 44100
STEP_SIZE_MS = 10
WINDOW_DURATION_MS = 100
PADDED_DURATION_MS = 400
NUM_MEL_BINS = 512
FREQ_MIN = 0
FREQ_MAX = 10000
POWER_FOR_IMAGE = 0.25
MAX_VALUE_SCALE_FACTOR = 65.5
GRIFFIN_LIM_ITERS = 300
CLIP_DURATION_MS = 5000

# Derived FFT parameters - matching Riffusion's calculations
N_FFT = int(PADDED_DURATION_MS / 1000.0 * SAMPLE_RATE)  # = 17640
WIN_LENGTH = int(WINDOW_DURATION_MS / 1000.0 * SAMPLE_RATE)  # = 4410
HOP_LENGTH = int(STEP_SIZE_MS / 1000.0 * SAMPLE_RATE)  # = 441

### 3.1 Convert Spectrogram to Audio

We define functions to reconstruct audio waveforms from spectrogram images using the inverse Mel scale and Griffin-Lim algorithm.

In [6]:
# Spectrogram to audio conversion

def wav_bytes_from_spectrogram_image(image: Image.Image) -> Tuple[io.BytesIO, float]:
    """Reconstruct audio using Riffusion's exact parameters"""

    # Get MAX_VALUE from EXIF data if available
    exif = image.getexif() if hasattr(image, 'getexif') else {}
    max_value = float(exif.get(11080, 46801012.0))  # Use images value or fall back to Riffusion default

    # Extract spectrogram from image
    img_data = np.array(image).astype(np.float32)

    # Flip Y and take first channel (as in Riffusion)
    # Handle both RGB and grayscale images
    if len(img_data.shape) == 3:  # RGB image
        spec_data = img_data[::-1, :, 0]
    else:  # Grayscale image
        spec_data = img_data[::-1, :]

    # Invert color (spectrograms are stored as inverted)
    spec_data = 255 - spec_data

    # Scale to original amplitude USING THE EXIF MAX_VALUE
    spec_data = spec_data * max_value / 255.0

    # Apply reverse power curve
    spec_data = np.power(spec_data + 1e-6, 1 / POWER_FOR_IMAGE)

    print(f"Processing spectrogram: shape={spec_data.shape}, using N_FFT={N_FFT}, HOP_LENGTH={HOP_LENGTH}")
    print(f"Using max_value={max_value} from image EXIF")

    # Convert to tensor
    device = "cuda" if torch.cuda.is_available() else "cpu"
    spec_tensor = torch.from_numpy(spec_data).to(device)

    try:
        # based on Riffusion's SpectrogramConverter.audio_from_spectrogram
        # Convert from mel scale to linear frequency
        inv_mel_scaler = torchaudio.transforms.InverseMelScale(
            n_mels=NUM_MEL_BINS,
            sample_rate=SAMPLE_RATE,
            f_min=FREQ_MIN,
            f_max=FREQ_MAX,
            n_stft=N_FFT // 2 + 1,
            norm=None,
            mel_scale="htk",
        ).to(device)

        # Convert to linear
        linear_spec = inv_mel_scaler(spec_tensor)

        # Run Griffin-Lim phase reconstruction
        griffin_lim = torchaudio.transforms.GriffinLim(
            n_fft=N_FFT,
            n_iter=GRIFFIN_LIM_ITERS,
            win_length=WIN_LENGTH,
            hop_length=HOP_LENGTH,
            power=1.0,
            momentum=0.99,
        ).to(device)

        # Generate waveform
        waveform = griffin_lim(linear_spec).cpu().numpy()

        print(f"Successfully reconstructed audio with correct parameters: {len(waveform)} samples")

        # Calculate expected duration
        duration_s = CLIP_DURATION_MS / 1000.0

        # soft clipping instead of hard normalization
        def soft_clip(x, threshold=0.8):
            """Soft clip values above threshold"""
            # gentle compression curve to values above threshold
            mask = np.abs(x) > threshold
            x[mask] = np.sign(x[mask]) * (threshold + (1 - threshold) *
                                          np.tanh((np.abs(x[mask]) - threshold) / (1 - threshold)))
            return x

        # Get the peak amplitude
        peak = np.max(np.abs(waveform))

        # Scale down only if necessary (above 0.95), otherwise preserve dynamics
        if peak > 0.95:
            waveform = waveform * (0.95 / peak)

        # Apply soft clipping to prevent any extreme values
        waveform = soft_clip(waveform)

        # Convert to int16
        waveform_int16 = (waveform * 32767).astype(np.int16)

        # Write WAV file
        output_bytes = io.BytesIO()
        wavfile.write(output_bytes, SAMPLE_RATE, waveform_int16)
        output_bytes.seek(0)

        return output_bytes, duration_s

    except Exception as e:
        print(f"Error in audio reconstruction: {e}")
        print("Using fallback method...")

        # Fallback to librosa if torchaudio fails
        try:
            # Use librosa with the same parameters
            from scipy.ndimage import zoom

            # Match n_fft//2+1 frequency bins for librosa
            target_freq_bins = N_FFT // 2 + 1
            if spec_data.shape[0] != target_freq_bins:
                print(f"Reshaping from {spec_data.shape[0]} to {target_freq_bins} frequency bins")
                zoom_factor = (target_freq_bins / spec_data.shape[0], 1)
                spec_data = zoom(spec_data, zoom_factor, order=1)

            # Run Griffin-Lim with same parameters
            waveform = librosa.griffinlim(
                spec_data,
                n_iter=GRIFFIN_LIM_ITERS,
                hop_length=HOP_LENGTH,
                win_length=WIN_LENGTH
            )

            # Normalize
            waveform = librosa.util.normalize(waveform) * 0.95

            # Create WAV bytes
            output_bytes = io.BytesIO()
            wavfile.write(output_bytes, SAMPLE_RATE, (waveform * 32767).astype(np.int16))
            output_bytes.seek(0)

            duration_s = len(waveform) / SAMPLE_RATE
            return output_bytes, duration_s

        except Exception as e2:
            print(f"Librosa fallback also failed: {e2}")

            # Last resort (from testing if it woudn't generate a signal): Generate simple audio
            print("Using basic sine wave as last resort")
            duration_s = CLIP_DURATION_MS / 1000.0
            t = np.linspace(0, duration_s, int(SAMPLE_RATE * duration_s))
            waveform = np.sin(2 * np.pi * 440 * t) * 0.5

            output_bytes = io.BytesIO()
            wavfile.write(output_bytes, SAMPLE_RATE, (waveform * 32767).astype(np.int16))
            output_bytes.seek(0)

            return output_bytes, duration_s


def spectrogram_from_image(
    image: Image.Image, max_volume: float = None, power_for_image: float = POWER_FOR_IMAGE
) -> np.ndarray:
    """
    Compute a spectrogram magnitude array from a spectrogram image.
    Enhanced with better error checking.
    """
    # Get MAX_VALUE from EXIF data if available
    if max_volume is None:
        exif = image.getexif() if hasattr(image, 'getexif') else {}
        max_volume = float(exif.get(11080, 46801012.0))  # Use Riffusion's default if not found

    # Convert to a numpy array of floats
    data = np.array(image).astype(np.float32)

    # Debug image data
    print(f"Image shape: {data.shape}, min: {np.min(data)}, max: {np.max(data)}")

    # Flip Y take a single channel
    data = data[::-1, :, 0]

    # Invert
    data = 255 - data

    # Rescale to max volume
    data = data * max_volume / 255

    # Add small epsilon to avoid zeros
    data = data + 1e-6

    # Reverse the power curve
    data = np.power(data, 1 / power_for_image)

    print(f"Final spectrogram shape: {data.shape}, min: {np.min(data)}, max: {np.max(data)}")

    return data

def waveform_from_spectrogram(
    Sxx: np.ndarray,
    n_fft: int = N_FFT,
    hop_length: int = HOP_LENGTH,
    win_length: int = WIN_LENGTH,
    num_samples: int = 0,
    sample_rate: int = SAMPLE_RATE,
    mel_scale: bool = True,
    n_mels: int = NUM_MEL_BINS,
    num_griffin_lim_iters: int = GRIFFIN_LIM_ITERS,
    device: str = "cpu",
) -> np.ndarray:
    """
    Robust spectrogram to audio conversion using librosa directly.
    """
    print(f"Input spectrogram shape: {Sxx.shape}")

    # Convert to numpy if it's a tensor
    if isinstance(Sxx, torch.Tensor):
        Sxx = Sxx.cpu().numpy()

    # Use librosa for the entire process to avoid torchaudio's InverseMelScale issues
    try:
        # Ensure we have the right shape for mel spectrogram
        if Sxx.shape[0] != n_mels and mel_scale:
            print(f"Reshaping spectrogram from {Sxx.shape[0]} to {n_mels} frequency bins")
            from scipy.ndimage import zoom
            zoom_factor = (n_mels / Sxx.shape[0], 1)
            Sxx = zoom(Sxx, zoom_factor, order=1)

        # Add a small epsilon to avoid numerical issues (nan/div by zero issues)
        Sxx = Sxx + 1e-8

        # If we're working with a mel spectrogram, convert directly with librosa
        if mel_scale:
            import librosa

            # First, convert power spectrogram to dB for better numerical stability
            S_db = librosa.power_to_db(Sxx, ref=np.max)

            # Clip to reasonable range to avoid extreme values
            S_db = np.clip(S_db, -80, 0)

            # Convert back to amplitude
            S_amplitude = librosa.db_to_amplitude(S_db)

            print(f"Using librosa Griffin-Lim directly")

            # Use Griffin-Lim algorithm on amplitude spectrogram
            waveform = librosa.griffinlim(
                S_amplitude,
                n_iter=num_griffin_lim_iters,
                hop_length=hop_length,
                win_length=win_length,
                random_state=42  # For reproducibility
            )
        else:
            # For regular spectrograms, just use Griffin-Lim directly
            waveform = librosa.griffinlim(
                Sxx,
                n_iter=num_griffin_lim_iters,
                hop_length=hop_length,
                win_length=win_length
            )

        print(f"Griffin-Lim complete, waveform length: {len(waveform)}")

    except Exception as e:
        print(f"Librosa approach failed: {e}")
        print("Falling back to basic sine wave generation")

        # Create a simple sound as backup if everything else fails
        duration = 5.0  # seconds
        waveform = np.zeros(int(sample_rate * duration))

        # Add a few sine waves at different frequencies
        t = np.linspace(0, duration, len(waveform))
        for freq in [440, 880, 1320]:  # A4, A5, E6
            waveform += 0.2 * np.sin(2 * np.pi * freq * t)

        # Apply envelope
        envelope = np.ones_like(waveform)
        fade_len = int(0.1 * sample_rate)  # 100ms fade
        envelope[:fade_len] = np.linspace(0, 1, fade_len)
        envelope[-fade_len:] = np.linspace(1, 0, fade_len)
        waveform *= envelope

    # Ensure consistent length
    if num_samples > 0:
        if len(waveform) > num_samples:
            waveform = waveform[:num_samples]
        else:
            waveform = np.pad(waveform, (0, max(0, num_samples - len(waveform))), mode='constant')

    # Normalize
    waveform = librosa.util.normalize(waveform) * 0.95

    print(f"Final waveform: length={len(waveform)}, min={np.min(waveform):.4f}, max={np.max(waveform):.4f}")

    return waveform

### 3.2 Define Text-to-Audio Generation Function

This function generates a spectrogram image from a text prompt using the model pipeline and converts it into playable audio.

In [7]:
# Generation function
def generate_audio(
    prompt: str,
    negative_prompt: str = "",
    num_inference_steps: int = 20,
    guidance_scale: float = 7.5,
    seed: int = None
) -> Tuple[Image.Image, io.BytesIO, float]:
    """
    Generate audio from text prompt.

    Parameters:
      prompt (str): Text prompt for generation
      negative_prompt (str): Negative prompt (concepts to avoid)
      num_inference_steps (int): Number of denoising steps
      guidance_scale (float): How closely to follow the prompt
      seed (int): Random seed for reproducibility

    Returns:
      spectrogram (Image.Image): Generated spectrogram image
      audio_bytes (io.BytesIO): Generated audio as bytes
      duration_s (float): Duration in seconds
    """
    # Set seed for reproducibility if provided
    generator = None
    if seed is not None:
        generator = torch.Generator("cpu").manual_seed(seed)

    # Generate spectrogram image
    start_time = time.time()
    spectrogram = pipe(
        prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator
    ).images[0]
    generation_time = time.time() - start_time

    # Convert to audio
    audio_bytes, duration_s = wav_bytes_from_spectrogram_image(spectrogram)

    print(f"Generation completed in {generation_time:.2f} seconds")
    print(f"Audio duration: {duration_s:.2f} seconds")

    return spectrogram, audio_bytes, duration_s

def display_generation(spectrogram, audio_bytes, prompt, negative_prompt=""):
    """Display the generated spectrogram and audio"""
    print(f"Prompt: {prompt}")
    if negative_prompt:
        print(f"Negative prompt: {negative_prompt}")

    # Display spectrogram
    plt.figure(figsize=(10, 5))
    plt.imshow(spectrogram)
    plt.title("Generated Spectrogram")
    plt.axis('off')
    plt.show()

    # Ensure we use the buffer properly
    audio_data = audio_bytes.getvalue() if hasattr(audio_bytes, 'getvalue') else audio_bytes
    # Play audio
    display(Audio(audio_data, rate=44100))  # Use the binary data directly

    return audio_bytes

### 3.3 Dataset Processing Functions

In [8]:
# Sample Processing Pipeline for Riffusion Fine-tuning
# import re for metadata extraction, prefix analysis etc...
import re

# Set random seed for reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Define dataset path
DATASET_PATH = '/content/drive/MyDrive/WAProd_Free_Anniversary_Collection_Vol7'
OUTPUT_PATH = '/content/drive/MyDrive/riffusion_finetuning'

# Create output directories
Path(OUTPUT_PATH).mkdir(exist_ok=True)
Path(os.path.join(OUTPUT_PATH, 'spectrograms')).mkdir(exist_ok=True)
Path(os.path.join(OUTPUT_PATH, 'metadata')).mkdir(exist_ok=True)



def verify_dataset_spectrograms(processed_files, sample_size=10):
    """Verify dataset spectrograms are in the correct format"""
    import random

    if len(processed_files) == 0:
        print("No files to verify")
        return False

    sample_files = random.sample(processed_files, min(sample_size, len(processed_files)))
    for file_info in sample_files:
        if 'spectrogram_path' not in file_info:
            continue

        # Load image and verify dimensions
        img_path = file_info['spectrogram_path']
        try:
            img = Image.open(img_path)
            # Check height (should be NUM_MEL_BINS)
            if img.height != NUM_MEL_BINS:
                print(f"WARNING: {img_path} has height {img.height}, expected {NUM_MEL_BINS}")

            # Check exif data
            exif = img.getexif()
            if 11040 not in exif or exif[11040] != NUM_MEL_BINS:
                print(f"WARNING: {img_path} missing proper metadata tags")
        except Exception as e:
            print(f"Error verifying {img_path}: {e}")

    print("Dataset verification complete")
    return True


def extract_metadata_from_path(file_path):
    """
    Extract metadata from file path and filename with improved pattern matching
    specifically optimized for WA-prefixed sample packs.
    """
    path = Path(file_path)
    filename = path.name
    path_str = str(file_path).lower()

    # Initialize metadata
    metadata = {
        'filename': filename,
        'path': str(file_path),
        'file_type': 'unknown',
        'genre': 'unknown',
        'bpm': None,
        'key': None,
        'is_loop': False,
        'is_oneshot': False
    }

    # 1. FILE TYPE DETECTION based on folder structure and filename
    # Common terms for categorizing sounds
    drum_terms = ['drum', 'kick', 'snare', 'clap', 'hat', 'cymbal', 'percussion', 'tom', 'perc', 'hihat', 'hi-hat', 'crash', 'ride']
    melodic_terms = ['bass', 'synth', 'melody', 'lead', 'pad', 'chord', 'arp', 'keys', 'piano']
    fx_terms = ['fx', 'effect', 'riser', 'sweep', 'impact', 'transition', 'downlifter', 'uplifter', 'build']
    vocal_terms = ['vocal', 'vox', 'voice', 'acapella']

    # Check filename first - most reliable
    name_lower = filename.lower()

    # Determine file type based on filename
    if any(term in name_lower for term in drum_terms):
        metadata['file_type'] = 'drum'
    elif any(term in name_lower for term in melodic_terms):
        metadata['file_type'] = 'melodic'
    elif any(term in name_lower for term in fx_terms):
        metadata['file_type'] = 'fx'
    elif any(term in name_lower for term in vocal_terms):
        metadata['file_type'] = 'vocal'

    # If not found in filename, check parent directories
    if metadata['file_type'] == 'unknown':
        if any(term in path_str for term in drum_terms):
            metadata['file_type'] = 'drum'
        elif any(term in path_str for term in melodic_terms):
            metadata['file_type'] = 'melodic'
        elif any(term in path_str for term in fx_terms):
            metadata['file_type'] = 'fx'
        elif any(term in path_str for term in vocal_terms):
            metadata['file_type'] = 'vocal'

    # 2. DETERMINE LOOP VS ONE-SHOT
    if 'loop' in name_lower or 'loops' in path_str:
        metadata['is_loop'] = True
    elif any(x in name_lower or x in path_str.lower() for x in ['shot', 'one shot', 'oneshot', 'hit']):
        metadata['is_oneshot'] = True

    # 3. EXTRACT BPM - Handle WA-style formats
    # Look for common patterns in your dataset
    bpm_patterns = [
        r'(\d+)(?:bpm|BPM)',                 # 140BPM
        r'_(\d+)bpm',                        # _140bpm
        r'_(\d+)_',                          # _140_
        r'_(\d+)\.',                         # _140.wav
        r'(\d+)BPM[_\.]',                    # 140BPM_
        r'BPM[_\s]*(\d+)',                   # BPM_140 or BPM 140
        r'[_\s](\d+)[_\s](?=.*loop)',        # Any number in a loop file
    ]

    for pattern in bpm_patterns:
        bpm_match = re.search(pattern, filename)
        if bpm_match:
            try:
                bpm_value = int(bpm_match.group(1))
                # Basic validation: most music is between 60-200 BPM
                if 60 <= bpm_value <= 200:
                    metadata['bpm'] = bpm_value
                    break
            except (ValueError, IndexError):
                pass

    # If BPM not found in filename, try path
    if metadata['bpm'] is None:
        for pattern in bpm_patterns:
            bpm_match = re.search(pattern, str(path))
            if bpm_match:
                try:
                    bpm_value = int(bpm_match.group(1))
                    # Basic validation: most music is between 60-200 BPM
                    if 60 <= bpm_value <= 200:
                        metadata['bpm'] = bpm_value
                        break
                except (ValueError, IndexError):
                    pass

    # 4. EXTRACT KEY - Handle various key notations including WASVT style
    # Look for common musical key patterns
    key_patterns = [
        r'([ABCDEFG][b#]?(?:m|min|maj)?)[_\s\.]',   # Matches A, Am, Abm, etc.
        r'([ABCDEFG][b#]?m?in)',                    # Matches Amin, A#min, etc.
        r'_([ABCDEFG][b#]?)[_\.]',                  # Matches _A_ or _A#. etc.
        r'_([ABCDEFG][b#]?m)[_\.]',                 # Matches _Am_ etc.
    ]

    for pattern in key_patterns:
        key_match = re.search(pattern, filename)
        if key_match:
            metadata['key'] = key_match.group(1)
            break

    # If key not found, look for common WA patterns with key in them
    if metadata['key'] is None and 'loop' in name_lower:
        # Try to extract from WASVT_Drum_Loop_Full_14_134BPM_C#m.wav format
        key_match = re.search(r'[_\s]([ABCDEFG][b#]?m)\.wav$', filename)
        if key_match:
            metadata['key'] = key_match.group(1)

    # 5. EXTRACT GENRE - Based on comprehensive genre list
    # Create mapping of genre indicators to standardized genres
    genre_mapping = {
        'edm': 'edm',
        'dubstep': 'dubstep', 'melodic dubstep': 'dubstep', 'future dubstep': 'dubstep', 'riddim dubstep': 'dubstep',
        'hardstyle': 'hardstyle', 'oracle': 'hardstyle',
        'bass house': 'bass_house', 'holy bass': 'bass_house', 'shark bass': 'bass_house',
        'brazilian bass': 'brazilian_bass',
        'slap house': 'slap_house',
        'techno': 'techno',
        'future bass': 'future_bass',
        'future pop': 'future_pop',
        'trap': 'trap', 'hybrid trap': 'trap', 'spicy vocal trap': 'trap',
        'lofi hip hop': 'lofi_hiphop', 'lofi': 'lofi_hiphop',
        'lofi cinematic': 'lofi_cinematic',
        'synthwave': 'synthwave', 'arcade': 'synthwave',
        'reggaeton': 'reggaeton',
        'cinematic': 'cinematic', 'movie': 'cinematic', 'horror': 'cinematic', 'cyberpunk': 'cinematic',
        'g house': 'g_house', 'g-house': 'g_house',
        'vocal': 'vocal_edm', 'vocals': 'vocal_edm', 'sinhala': 'vocal_edm'
    }

    # Check for genre in path string
    path_lower = path_str.lower()
    for indicator, genre in genre_mapping.items():
        if indicator in path_lower:
            metadata['genre'] = genre
            break

    # Map WA prefixes to genres (based on your dataset patterns)
    wa_pattern = re.search(r'(WA[A-Z]+)_', filename)
    if wa_pattern:
        prefix = wa_pattern.group(1)
        wa_genre_map = {
            'WATR': 'trap',
            'WAED': 'edm',
            'WADB': 'dubstep',
            'WABB': 'boom_bap',
            'WAGHC': 'hardcore',
            'WALE': 'electronic',
            'WALC': 'lofi_hiphop',
            'WADHF': 'deep_house',
            'WAFHS': 'future_house',
            'WASVT': 'synthwave',
            'WASA': 'ambient',
            'WART': 'retro',
            'WATE': 'techno',
            'WAHTP': 'hip_hop',
            'WARDB': 'riddim_dubstep',
            'WABFH': 'bass_house',
            'WAFDE': 'future_bass',
            'WAFDJ': 'future_bass',
            'WAHBH': 'hardstyle',
            'WALHHF': 'lofi_hiphop',
        }
        if prefix in wa_genre_map:
            metadata['genre'] = wa_genre_map[prefix]

    return metadata

def analyze_wa_prefixes(all_files):
    """Analyze WA prefixes in the dataset to identify patterns"""
    prefixes = {}

    for file_info in all_files:
        filename = file_info['filename']
        wa_match = re.search(r'(WA[A-Z]+)_', filename)
        if wa_match:
            prefix = wa_match.group(1)
            if prefix not in prefixes:
                prefixes[prefix] = {
                    'count': 0,
                    'file_types': set(),
                    'bpms': set(),
                    'keys': set()
                }

            prefixes[prefix]['count'] += 1
            if file_info['file_type'] != 'unknown':
                prefixes[prefix]['file_types'].add(file_info['file_type'])
            if file_info.get('bpm'):
                prefixes[prefix]['bpms'].add(file_info['bpm'])
            if file_info.get('key'):
                prefixes[prefix]['keys'].add(file_info['key'])

    # Print analysis
    print("\nWA Prefix Analysis:")
    for prefix, data in sorted(prefixes.items(), key=lambda x: x[1]['count'], reverse=True):
        print(f"{prefix}: {data['count']} files")
        print(f"  File types: {', '.join(data['file_types']) if data['file_types'] else 'unknown'}")
        print(f"  BPM range: {min(data['bpms']) if data['bpms'] else 'N/A'} - {max(data['bpms']) if data['bpms'] else 'N/A'}")
        print(f"  Keys: {', '.join(sorted(list(data['keys']))) if data['keys'] else 'N/A'}")

    return prefixes

def scan_dataset(root_path, limit_per_type=None, max_files=None):
    """
    Scan the dataset and extract metadata with more comprehensive statistics

    Args:
        root_path: Path to the dataset directory
        limit_per_type: Maximum files per type (optional, None for no limit)
        max_files: Maximum total files to process (optional, None for no limit)

    Returns:
        List of metadata dictionaries and statistics summary
    """
    print(f"Scanning {root_path} for audio files...")
    all_files = []
    type_counters = {'drum': 0, 'melodic': 0, 'fx': 0, 'vocal': 0, 'unknown': 0}
    genre_counters = {}
    key_counters = {}
    bpm_values = []
    total_wavs_found = 0

    # Collect all WAV files first
    wav_files = []
    for root, _, files in os.walk(root_path):
        for file in files:
            if file.lower().endswith('.wav'):
                wav_files.append(os.path.join(root, file))
                total_wavs_found += 1

                # Stop if we hit the max files limit
                if max_files and total_wavs_found >= max_files:
                    break

        if max_files and total_wavs_found >= max_files:
            break

    print(f"Found {total_wavs_found} WAV files in total")

    # Process files with metadata extraction
    for file_path in tqdm(wav_files):
        # Extract metadata
        metadata = extract_metadata_from_path(file_path)
        file_type = metadata['file_type']

        # Count by type for balanced sampling
        if limit_per_type and type_counters.get(file_type, 0) >= limit_per_type:
            continue

        # Update counters
        type_counters[file_type] = type_counters.get(file_type, 0) + 1

        # Update genre statistics
        genre = metadata.get('genre', 'unknown')
        genre_counters[genre] = genre_counters.get(genre, 0) + 1

        # Update key statistics
        key = metadata.get('key')
        if key:
            key_counters[key] = key_counters.get(key, 0) + 1

        # Update BPM statistics
        bpm = metadata.get('bpm')
        if bpm:
            bpm_values.append(bpm)

        all_files.append(metadata)

    # Generate statistics summary
    stats = {
        'total_samples': len(all_files),
        'file_type_distribution': type_counters,
        'genre_distribution': genre_counters,
        'key_distribution': key_counters,
        'bpm_stats': {
            'count': len(bpm_values),
            'min': min(bpm_values) if bpm_values else None,
            'max': max(bpm_values) if bpm_values else None,
            'mean': sum(bpm_values)/len(bpm_values) if bpm_values else None,
        }
    }

    print(f"Selected {len(all_files)} files for processing")
    print(f"Type distribution: {type_counters}")
    print(f"Top genres: {sorted(genre_counters.items(), key=lambda x: x[1], reverse=True)[:5]}")
    if bpm_values:
        print(f"BPM range: {stats['bpm_stats']['min']} - {stats['bpm_stats']['max']} (avg: {stats['bpm_stats']['mean']:.1f})")
    else:
        print("No BPM information found")
    print(f"Unique keys found: {len(key_counters)}")

    return all_files, stats




# Function to handle longer audio files
def preprocess_audio_file(audio_path, max_duration_ms=5000):
    """Preprocess a single audio file, splitting if needed"""
    try:
        # Load audio
        segment = pydub.AudioSegment.from_file(audio_path)
        duration_ms = len(segment)

        # For short files, just return the original path
        if duration_ms <= max_duration_ms:
            return [audio_path]

        # For long files, split into clips
        clips = []
        temp_dir = os.path.join(OUTPUT_PATH, "temp_clips")
        os.makedirs(temp_dir, exist_ok=True)

        # Determine number of clips
        num_clips = max(1, duration_ms // max_duration_ms)

        for i in range(num_clips):
            if i == 0:
                start_ms = 0
            else:
                start_ms = i * (duration_ms - max_duration_ms) // max(1, num_clips - 1)

            # Extract clip
            clip = segment[start_ms:start_ms + max_duration_ms]

            # Save to temp file
            output_path = os.path.join(temp_dir, f"{Path(audio_path).stem}_clip_{i}.wav")
            clip.export(output_path, format="wav")
            clips.append(output_path)

        return clips

    except Exception as e:
        print(f"Error preprocessing {audio_path}: {e}")
        return [audio_path]  # Return original as fallback



def process_samples(file_list, output_dir, max_samples=None):
    """
    Process audio files into spectrograms, skipping already processed files.

    Args:
        file_list: List of file metadata dictionaries
        output_dir: Directory to save spectrograms
        max_samples: Maximum samples to process (None for all)
    """
    # Count already processed files
    already_processed = 0
    processed_files = []

    # Shuffle and limit samples
    if max_samples and len(file_list) > max_samples:
        random.shuffle(file_list)
        file_list = file_list[:max_samples]

    print(f"Processing {len(file_list)} audio files...")

    for file_info in tqdm(file_list):
        audio_path = file_info['path']
        file_type = file_info['file_type']

        # Preprocess audio (split if too long)
        audio_clips = preprocess_audio_file(audio_path)

        for i, clip_path in enumerate(audio_clips):
            # Add clip suffix for multiple clips
            if len(audio_clips) > 1:
                filename = f"{Path(audio_path).stem}_clip_{i}"
            else:
                filename = Path(audio_path).stem

            # Create directory structure if needed
            type_dir = os.path.join(output_dir, file_type)
            os.makedirs(type_dir, exist_ok=True)

            # Save to subdirectory by file type
            spectrogram_path = os.path.join(type_dir, f"{filename}.png")

            # Check if file already exists before processing
            if os.path.exists(spectrogram_path):
                # File exists, skip regeneration
                already_processed += 1

                # Create a copy of metadata for this clip
                clip_info = file_info.copy()
                clip_info['spectrogram_path'] = spectrogram_path
                processed_files.append(clip_info)

                # Only print occasionally to avoid console spam
                if already_processed % 100 == 0:
                    print(f"Found {already_processed} existing spectrograms")
            else:
                # Process only if needed
                try:
                    result = audio_to_spectrogram(clip_path, spectrogram_path)
                    if result:
                        # Create a copy of metadata for this clip
                        clip_info = file_info.copy()
                        clip_info['spectrogram_path'] = spectrogram_path
                        processed_files.append(clip_info)
                except Exception as e:
                    print(f"Error processing {clip_path}: {e}")

    print(f"Used {already_processed} existing spectrograms, processed {len(processed_files) - already_processed} new files")

    # Save metadata
    metadata_path = os.path.join(output_dir, "metadata.json")
    with open(metadata_path, 'w') as f:
        json.dump(processed_files, f, indent=2)

    return processed_files

def analyze_dataset(processed_files):
    """Print statistics about the dataset"""
    print("Dataset statistics:")
    print(f"Total samples: {len(processed_files)}")

    if not processed_files:
        print("No samples to analyze")
        return

    # Check if processed_files is a list of dictionaries

    try:
        # Make sure we have a list of dictionaries
        if isinstance(processed_files, list) and all(isinstance(item, dict) for item in processed_files):
            df = pd.DataFrame(processed_files)

            # Print file type distribution if available
            if 'file_type' in df.columns:
                print("\nFile type distribution:")
                print(df['file_type'].value_counts())

            # Print genre distribution if available
            if 'genre' in df.columns:
                valid_genres = df[df['genre'] != 'unknown']
                if not valid_genres.empty:
                    print("\nGenre distribution:")
                    print(valid_genres['genre'].value_counts())

            # Print BPM statistics if available
            if 'bpm' in df.columns:
                valid_bpm = df[df['bpm'].notna()]
                if not valid_bpm.empty:
                    print(f"\nBPM statistics:")
                    print(f"Min BPM: {valid_bpm['bpm'].min()}")
                    print(f"Max BPM: {valid_bpm['bpm'].max()}")
                    print(f"Mean BPM: {valid_bpm['bpm'].mean():.1f}")

            # Print key distribution if available
            if 'key' in df.columns:
                valid_keys = df[df['key'].notna()]
                if not valid_keys.empty:
                    print(f"\nKey distribution:")
                    print(valid_keys['key'].value_counts())
        else:
            print("Warning: processed_files is not in the expected format (list of dictionaries)")
            return

    except Exception as e:
        print(f"Error analyzing dataset: {e}")
        print("Performing basic analysis instead...")

        # Fallback to basic analysis without pandas
        file_types = {}
        genres = {}
        keys = {}
        bpms = []

        for file in processed_files:
            if isinstance(file, dict):
                ft = file.get('file_type', 'unknown')
                file_types[ft] = file_types.get(ft, 0) + 1

                genre = file.get('genre', 'unknown')
                genres[genre] = genres.get(genre, 0) + 1

                key = file.get('key')
                if key:
                    keys[key] = keys.get(key, 0) + 1

                bpm = file.get('bpm')
                if bpm:
                    bpms.append(bpm)

        print("\nFile type counts:", file_types)
        print("Top genres:", sorted(genres.items(), key=lambda x: x[1], reverse=True)[:5])
        print("Top keys:", sorted(keys.items(), key=lambda x: x[1], reverse=True)[:5])
        if bpms:
            print(f"BPM range: {min(bpms)} - {max(bpms)} (avg: {sum(bpms)/len(bpms):.1f})")


def find_wav_files(root_path, limit=10):
    """Simple function to find WAV files directly"""
    wav_files = []

    print(f"Looking for WAV files in {root_path}...")

    for root, _, files in os.walk(root_path):
        for file in files:
            if file.lower().endswith('.wav'):
                wav_path = os.path.join(root, file)
                wav_files.append(wav_path)
                print(f"Found: {wav_path}")

                if len(wav_files) >= limit:
                    return wav_files

    return wav_files


# Helper functions
def process_dataset_pipeline(dataset_path, output_path):
    """Orchestrates the complete dataset processing pipeline"""
    # Find sample files first
    sample_wavs = find_wav_files(dataset_path, limit=10)
    if not sample_wavs:
        print("No WAV files found in the dataset path. Please check the path.")
        return None, None, None

    # Run test on single file
    test_result = run_single_file_test(sample_wavs[0], output_path)
    if not test_result:
        print("Test conversion failed - check audio processing functions")
        return None, None, None

    # Process full dataset
    print("\n Processing complete dataset...")
    all_files, stats = extract_dataset_metadata(dataset_path)

    return all_files, stats, sample_wavs

def extract_dataset_metadata(dataset_path):
    """Extract metadata from all files in dataset"""
    try:
        all_files_data = scan_dataset(dataset_path)
        all_files = all_files_data[0]
        stats = all_files_data[1] if len(all_files_data) > 1 else None

        # Fallback to direct processing if needed
        if len(all_files) < 10:
            print("Using direct file processing method...")
            all_files = direct_file_processing(dataset_path)

        return all_files, stats
    except Exception as e:
        print(f"Error in metadata extraction: {e}")
        return [], None

def direct_file_processing(dataset_path):
    """Direct WAV file processing as fallback"""
    all_wav_files = []
    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith('.wav'):
                file_path = os.path.join(root, file)
                metadata = extract_metadata_from_path(file_path)
                all_wav_files.append(metadata)

    print(f"Processed {len(all_wav_files)} files directly")
    return all_wav_files

def run_single_file_test(test_file, output_path):
    """Test spectrogram conversion on a single file"""
    print("\n Testing conversion on sample file...")
    test_output = os.path.join(output_path, "test_spectrogram.png")
    result = audio_to_spectrogram(test_file, test_output)

    if result:
        print(f"Test successful: {test_output}")
        display(Image.open(test_output))
        return True
    else:
        print("Test failed")
        return False

def process_and_prepare_training_data(all_files, output_path):
    """Process spectrograms and prepare training data, checking for existing data first"""
    if not all_files:
        print("No files to process")
        return None

    # Check for existing training data first
    training_data_file = os.path.join(output_path, 'metadata', 'full_dataset_metadata.csv')
    training_json_file = os.path.join(output_path, "training_data.json")

    if os.path.exists(training_json_file):
        print(f"Using existing training data from {training_json_file}")
        with open(training_json_file, 'r') as f:
            training_data = json.load(f)

        # Verify all files still exist
        valid_entries = []
        missing_files = 0

        for entry in training_data:
            if 'image_path' in entry and os.path.exists(entry['image_path']):
                valid_entries.append(entry)
            else:
                missing_files += 1

        if missing_files > 0:
            print(f"Warning: {missing_files} files from existing training data are missing")

        if valid_entries:
            print(f"Loaded {len(valid_entries)} valid entries from existing training data")
            return valid_entries
        else:
            print("No valid entries in existing training data, regenerating...")

    # Create spectrograms
    print("\nGenerating spectrograms...")
    spectrogram_dir = os.path.join(output_path, 'spectrograms')
    processed_files = process_samples(all_files, spectrogram_dir)

    if not processed_files:
        print("Failed to process files into spectrograms")
        return None

    # Save metadata
    metadata_df = pd.DataFrame(processed_files)
    metadata_path = os.path.join(output_path, 'metadata', 'full_dataset_metadata.csv')
    os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
    metadata_df.to_csv(metadata_path, index=False)
    print(f"Saved complete metadata to {metadata_path}")

    # Prepare training data
    print("\nPreparing data for fine-tuning...")
    training_data = prepare_for_lora_finetuning(processed_files, output_path)

    if training_data:
        # Visualize example spectrograms
        visualize_spectrograms(training_data, num_examples=5)
        print(f"Pipeline completed successfully! Processed {len(processed_files)} files.")
        verify_dataset_spectrograms(processed_files)
        return training_data
    else:
        print("Failed to create training data")
        return None

### 3.4 Spectrogram Conversion Pipeline

In [ ]:
def audio_to_spectrogram(audio_path, output_path=None):
    """Generate spectrograms exactly matching Riffusion's approach"""
    try:
        # 1. LOAD AUDIO
        audio_segment = pydub.AudioSegment.from_file(audio_path)

        # 2. ENSURE MONO
        if audio_segment.channels > 1:
            audio_segment = audio_segment.set_channels(1)

        # 3. ENSURE SAMPLE RATE
        if audio_segment.frame_rate != SAMPLE_RATE:
            audio_segment = audio_segment.set_frame_rate(SAMPLE_RATE)

        # 4. CONVERT TO NUMPY ARRAY
        samples = np.array(audio_segment.get_array_of_samples()).astype(np.float32)
        samples = samples / 32768.0  # Convert to float [-1, 1]

        # 5. APPLY PREEMPHASIS
        samples = librosa.effects.preemphasis(samples, coef=0.97)

        # 6. PRECISE SAMPLE TRIMMING
        n_fft = N_FFT
        hop_length = HOP_LENGTH
        target_time_frames = 568
        target_samples = (target_time_frames - 1) * hop_length + n_fft

        if len(samples) > target_samples:
            excess = len(samples) - target_samples
            start = excess // 2
            samples = samples[start:start + target_samples]
        else:
            pad_length = target_samples - len(samples)
            pad_left = pad_length // 2
            pad_right = pad_length - pad_left
            samples = np.pad(samples, (pad_left, pad_right), mode='constant')

        # 7. COMPUTE STFT
        D = librosa.stft(
            samples,
            n_fft=n_fft,
            hop_length=hop_length,
            win_length=WIN_LENGTH,
            window='hann',
            center=False,
            dtype=np.complex64,
            pad_mode='constant'
        )

        # 8. USE MAGNITUDE
        S = np.abs(D)

        # 9. APPLY BASS BOOST BEFORE MEL CONVERSION
        # This enhances low frequencies that might get lost in mel conversion
        freq_bins = S.shape[0]
        bass_range = int(freq_bins * 0.1)  # Boost bottom 10%
        bass_boost = np.ones(freq_bins)
        bass_boost[:bass_range] = np.linspace(1.8, 1.0, bass_range)
        S = S * bass_boost[:, np.newaxis]

        # 10. MEL CONVERSION
        S_mel = librosa.feature.melspectrogram(
            S=S,  # Using magnitude spectrogram
            sr=SAMPLE_RATE,
            n_fft=n_fft,
            n_mels=NUM_MEL_BINS,
            fmin=FREQ_MIN,
            fmax=FREQ_MAX,
            htk=True,
            norm='slaney'
        )

        # 11. STORE MAX VALUE
        max_value = np.max(S_mel)
        RIFFUSION_MAX_VALUE = max_value * MAX_VALUE_SCALE_FACTOR

        # 12. RIFFUSION-STYLE NORMALIZATION
        # Normalize by max
        S_norm = S_mel / (max_value + 1e-8)

        # 13. APPLY POWER CURVE (like Riffusion)
        S_norm = np.power(S_norm, POWER_FOR_IMAGE)

        # 14. SCALE TO 0-255
        S_norm = S_norm * 255

        # 15. INVERT
        S_norm = 255 - S_norm

        # 16. FLIP FREQUENCY AXIS
        S_norm = S_norm[::-1, :]

        # 17. VERIFY DIMENSIONS
        if S_norm.shape[1] != target_time_frames:
            if S_norm.shape[1] < target_time_frames:
                S_norm = np.pad(S_norm, ((0, 0), (0, target_time_frames - S_norm.shape[1])),
                               mode='constant', constant_values=255)
            else:
                S_norm = S_norm[:, :target_time_frames]

        # 18. CREATE IMAGE
        img_data = np.stack([S_norm, S_norm, S_norm], axis=-1).astype(np.uint8)
        img = Image.fromarray(img_data, 'RGB')

        # 19. ADD RIFFUSION METADATA
        exif = img.getexif()
        exif[11000] = SAMPLE_RATE
        exif[11010] = STEP_SIZE_MS
        exif[11040] = NUM_MEL_BINS
        exif[11050] = FREQ_MIN
        exif[11060] = FREQ_MAX
        exif[11070] = POWER_FOR_IMAGE
        exif[11080] = RIFFUSION_MAX_VALUE

        # 20. SAVE IMAGE
        if output_path:
            os.makedirs(os.path.dirname(output_path), exist_ok=True)
            img.save(output_path, exif=exif)
            return output_path
        else:
            return img
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        traceback.print_exc()
        return None

def spectrogram_to_audio(spectrogram_path, output_path=None):
    """Convert spectrogram image back to audio; Simplified version of Riffusion's reconstruction"""
    try:
        # Load image
        img = Image.open(spectrogram_path)
        img_data = np.array(img)

        # Convert to grayscale and normalize
        S_norm = img_data[:, :, 0].astype(np.float32) / 255.0
        S_norm = 1.0 - S_norm  # Undo inversion

        # Scale to dB
        min_db = -80
        max_db = 0
        S_db = S_norm * (max_db - min_db) + min_db

        # Convert to magnitude
        S = librosa.db_to_power(S_db)

        # Griffin-Lim algorithm to recover phase
        y = librosa.griffinlim(
            S,
            n_iter=32,
            hop_length=512,
            win_length=2048
        )

        # Save audio if output path is provided
        if output_path:
            librosa.output.write_wav(output_path, y, sr=44100, norm=True)
            return output_path

        return y, 44100

    except Exception as e:
        print(f"Error processing {spectrogram_path}: {e}")
        return None

def prepare_for_lora_finetuning(processed_files, output_dir):
    """Create training data with prompt-image pairs"""
    training_data = []

    if not processed_files:
        print("No processed files to prepare for training")
        return []

    for file_info in processed_files:
        if 'spectrogram_path' not in file_info:
            continue

        # Create prompt based on metadata
        prompt_parts = []

        # Add file type
        file_type = file_info.get('file_type', 'unknown')
        if file_type == 'drum':
            prompt_parts.append("drum sound")
        elif file_type == 'melodic':
            prompt_parts.append("melodic sound")
        elif file_type == 'fx':
            prompt_parts.append("sound effect")
        elif file_type == 'vocal':
            prompt_parts.append("vocal sample")
        else:
            prompt_parts.append("audio sample")

        # Add genre if known
        genre = file_info.get('genre', 'unknown')
        if genre != 'unknown':
            prompt_parts.append(genre)

        # Add BPM if available
        bpm = file_info.get('bpm')
        if bpm:
            prompt_parts.append(f"{bpm} BPM")

        # Add key if available
        key = file_info.get('key')
        if key:
            prompt_parts.append(f"in key {key}")

        # Filename might contain useful info
        filename = file_info.get('filename', '')
        clean_name = re.sub(r'[_\d]+', ' ', Path(filename).stem)
        clean_name = re.sub(r'\s+', ' ', clean_name).strip()
        if clean_name and len(clean_name) > 3:
            prompt_parts.append(clean_name)

        # Combine into a prompt
        prompt = " ".join(prompt_parts)

        training_data.append({
            "prompt": prompt,
            "image_path": file_info['spectrogram_path'],
            "metadata": file_info
        })

    # Save the training data
    training_file = os.path.join(output_dir, "training_data.json")
    with open(training_file, 'w') as f:
        json.dump(training_data, f, indent=2)

    print(f"Created training data with {len(training_data)} entries")

    # Show a few examples
    if training_data:
        print("\nExample prompts:")
        for i in range(min(5, len(training_data))):
            print(f"  {i+1}: {training_data[i]['prompt']}")

    return training_data

# Visualize spectrograms
def visualize_spectrograms(training_data, num_examples=5):
    """Visualize sample spectrograms"""
    if not training_data:
        print("No training data to visualize")
        return

    samples = random.sample(training_data, min(num_examples, len(training_data)))

    fig, axes = plt.subplots(len(samples), 1, figsize=(10, 4*len(samples)))
    if len(samples) == 1:
        axes = [axes]

    for i, sample in enumerate(samples):
        try:
            img = Image.open(sample['image_path'])
            axes[i].imshow(img)
            axes[i].set_title(f"Prompt: {sample['prompt']}")
            axes[i].axis('off')
        except Exception as e:
            print(f"Error displaying image {sample['image_path']}: {e}")

    plt.tight_layout()
    plt.show()

"""If Dataset must be reprocesssed, run this cell to delete the most recently processed dataset (converted spectrograms)"""
# import shutil
# # Be careful with this as it will delete all processed spectrograms
# spectrograms_dir = os.path.join(OUTPUT_PATH, 'spectrograms')
# if os.path.exists(spectrograms_dir):
#     shutil.rmtree(spectrograms_dir)
# os.makedirs(spectrograms_dir, exist_ok=True)


# Verify we can find some WAV files
sample_wavs = find_wav_files(DATASET_PATH, limit=10)
print(f"Found {len(sample_wavs)} sample WAV files")

if sample_wavs:
    # process these directly
    print("\nTesting spectrogram conversion on one file...")
    test_file = sample_wavs[0]
    test_output = os.path.join(OUTPUT_PATH, "test_spectrogram.png")
    result = audio_to_spectrogram(test_file, test_output)

    if result:
        print(f"Successfully created spectrogram: {test_output}")
        # Display the test spectrogram
        display(Image.open(test_output))
    else:
        print("Failed to create test spectrogram")

    # If that works, proceed with the full pipeline
    print("\n Now running the full pipeline...")
    # Process the entire dataset without limits

# Process dataset
all_files, stats, sample_wavs = process_dataset_pipeline(DATASET_PATH, OUTPUT_PATH)

# Analyze dataset if processing succeeded
if all_files:
    print("\n Dataset Analysis:")
    analyze_dataset(all_files)

    # Process spectrograms and generate training data
    process_and_prepare_training_data(all_files, OUTPUT_PATH)

### 3.5 Dataset Analysis and Visualization

In [ ]:
# Import Seaborn for better plots
import seaborn as sns

def visualize_wa_prefixes(all_files):
    """Visualize WA prefix distribution and characteristics"""
    # Get WA prefix analysis
    prefixes = analyze_wa_prefixes(all_files)

    # Create a DataFrame for plotting
    prefix_data = []
    for prefix, data in prefixes.items():
        prefix_data.append({
            'prefix': prefix,
            'total_count': data['count'],
            'file_types': len(data['file_types']),
            'bpm_count': len(data['bpms']),
            'key_count': len(data['keys'])
        })

    if not prefix_data:
        print("No WA prefixes found to visualize")
        return

    df = pd.DataFrame(prefix_data)

    # Plot prefix distribution
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df.sort_values('total_count', ascending=False).head(15),
                x='prefix', y='total_count')
    plt.title('Top 15 WA Prefixes by Count')
    plt.xticks(rotation=45)
    plt.xlabel('Prefix')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

    # Plot prefix characteristics
    plt.figure(figsize=(14, 6))

    # Reshape data for grouped bar chart
    melt_df = pd.melt(df.sort_values('total_count', ascending=False).head(10),
                       id_vars=['prefix'],
                       value_vars=['file_types', 'bpm_count', 'key_count'],
                       var_name='metric', value_name='value_count')

    # Plot characteristics of top 10 prefixes
    sns.barplot(data=melt_df, x='prefix', y='value_count', hue='metric')
    plt.title('Characteristics of Top 10 WA Prefixes')
    plt.xticks(rotation=45)
    plt.xlabel('Prefix')
    plt.ylabel('Count')
    plt.legend(title='Metric')
    plt.tight_layout()
    plt.show()

def visualize_metadata_stats(all_files):
    """Create visualizations for the extracted metadata"""
    # Convert to DataFrame for easier analysis
    df = pd.DataFrame(all_files)

    # Create a figure with subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # 1. File Type Distribution
    if 'file_type' in df.columns:
        sns.countplot(y=df['file_type'], ax=axes[0, 0])
        axes[0, 0].set_title('File Type Distribution')
        axes[0, 0].set_ylabel('')

    # 2. Genre Distribution (top 10)
    if 'genre' in df.columns:
        genre_counts = df['genre'].value_counts()
        top_genres = genre_counts.head(10)
        sns.barplot(x=top_genres.values, y=top_genres.index, ax=axes[0, 1])
        axes[0, 1].set_title('Top 10 Genres')
        axes[0, 1].set_xlabel('Count')
        axes[0, 1].set_ylabel('')

    # 3. BPM Distribution
    if 'bpm' in df.columns:
        bpm_data = df[df['bpm'].notna()]['bpm']
        if not bpm_data.empty:
            sns.histplot(bpm_data, kde=True, ax=axes[1, 0])
            axes[1, 0].set_title('BPM Distribution')
            axes[1, 0].set_xlabel('BPM')

    # 4. Key Distribution (top 10)
    if 'key' in df.columns:
        key_counts = df['key'].value_counts()
        top_keys = key_counts.head(10)
        sns.barplot(x=top_keys.values, y=top_keys.index, ax=axes[1, 1])
        axes[1, 1].set_title('Top 10 Musical Keys')
        axes[1, 1].set_xlabel('Count')
        axes[1, 1].set_ylabel('')

    plt.tight_layout()
    plt.show()

    return df


print("Scanning and analyzing dataset...")
all_files, stats = scan_dataset(DATASET_PATH, limit_per_type=1000, max_files=None)

# Save metadata to file for later use
metadata_df = pd.DataFrame(all_files)
metadata_df.to_csv(os.path.join(OUTPUT_PATH, 'metadata', 'dataset_metadata.csv'), index=False)

def generate_report_visualizations(all_files, stats, output_path):
    """Generate report-ready visualizations and exports"""
    # Create folder for report assets
    report_dir = os.path.join(output_path, 'report_assets')
    os.makedirs(report_dir, exist_ok=True)

    # Generate and save primary visualizations
    print("\n Generating report visualizations...")
    df = visualize_metadata_stats(all_files)

    # Export summary statistics
    stats_df = pd.DataFrame({
        'Metric': ['Total Samples', 'Genres', 'Min BPM', 'Max BPM', 'Avg BPM', 'Keys'],
        'Value': [
            stats['total_samples'],
            len(stats['genre_distribution']),
            stats['bpm_stats']['min'] if stats['bpm_stats']['count'] > 0 else 'N/A',
            stats['bpm_stats']['max'] if stats['bpm_stats']['count'] > 0 else 'N/A',
            f"{stats['bpm_stats']['mean']:.1f}" if stats['bpm_stats']['count'] > 0 else 'N/A',
            len(stats['key_distribution'])
        ]
    })
    stats_df.to_csv(os.path.join(report_dir, 'summary_stats.csv'), index=False)

    # Generate report summary
    print("\n Dataset Summary:")
    summary = f"""
    The dataset contains {stats['total_samples']} audio samples across {len(stats['genre_distribution'])}
    different musical genres. File types include {', '.join(stats['file_type_distribution'].keys())} with
    BPM ranging from {stats['bpm_stats']['min']} to {stats['bpm_stats']['max']} and
    {len(stats['key_distribution'])} distinct musical keys.
    """
    print(summary)

    # Save summary to text file
    with open(os.path.join(report_dir, 'dataset_summary.txt'), 'w') as f:
        f.write(summary)

    print(f"Report assets saved to {report_dir}")

    return df

generate_report_visualizations(all_files, stats, OUTPUT_PATH)

## 4. Example Output and Testing (pre-fine tuning)

Here, we test the generation pipeline with a sample input prompt and visualize the spectrogram alongside the synthesized audio.

### 4.1 Generate Audio from Sample Prompt (Riffusion Pre-Fine Tuning)

We run the riffusion pipeline using an example text prompt and display both the spectrogram and audio for evaluation.

In [ ]:
# Generate example audio from Riffusion model
def run_all_examples():
    examples = [
        ('Jazz Piano Solo', """smooth jazz piano solo, sophisticated chord progression, professional recording, clear audio""", """drums, vocals, noise, distortion, low quality, muffled""", 50, 42),
        ('Electronic Dance Music', """energetic electronic dance beat, heavy bass, synthesizer leads, club music, professional production""", """acoustic instruments, classical, slow tempo, ambient, voices""", 40, 123),
        ('Classical Violin', """classical violin concerto, orchestral accompaniment, romantic period, beautiful melody, concert hall recording""", """electric guitar, drums, electronic music, distortion, modern production""", 60, 888),
        ('Ambient Nature Sounds', """peaceful forest ambience, gentle rain, birds chirping, nature sounds, relaxing atmosphere""", """music, instruments, voices, urban sounds, loud noise""", 30, 555),
        ('Rock Guitar Riff', """heavy electric guitar riff, rock music, power chords, distorted guitar, band practice room""", """piano, classical, ambient, electronic, clean tone""", 45, 999),
        ('Multi-instrument Jazz', """jazz trio with piano, upright bass, and brush drums, sophisticated harmony, live club recording""", """electric guitar, vocals, synthesizer, distortion, low quality recording""", 55, 777)
    ]

    for i, (name, prompt, neg_prompt, steps, seed) in enumerate(examples):
        print(f"\n===== {i+1}. {name} =====")

        spectrogram, audio_bytes, duration = generate_audio(
            prompt=prompt,
            negative_prompt=neg_prompt,
            num_inference_steps=steps,
            seed=seed
        )

        audio_data = display_generation(spectrogram, audio_bytes, prompt, neg_prompt)

        yield name, audio_data, duration

# Run all examples
results = list(run_all_examples())

# Save to file
# with open("output.wav", "wb") as f:
#     f.write(audio_bytes.getbuffer())
# print("Audio saved to output.wav")

## 5. Fine-tuning Implementation

### 5.1 Dataset and Configuration Classes
LoRA configuration and dataset classes

In [11]:
# dataclass utils for configuration
from dataclasses import dataclass, field

def custom_collate_fn(batch):
    """More robust collate function for audio spectrograms"""
    # Filter out None and empty values
    batch = [item for item in batch if item is not None and isinstance(item, dict)]

    if len(batch) == 0:
        return None

    # Resize any images with inconsistent dimensions
    for item in batch:
        if 'image' in item and item['image'] is not None and isinstance(item['image'], torch.Tensor):
            # Check if dimensions are wrong
            if item['image'].shape[1] == NUM_MEL_BINS and item['image'].shape[2] != 512:
                # Resize to consistent width
                item['image'] = torch.nn.functional.interpolate(
                    item['image'].unsqueeze(0),  # Add batch dimension
                    size=(NUM_MEL_BINS, 512),  # Target size
                    mode='bilinear',
                    align_corners=False
                ).squeeze(0)  # Remove batch dimension

    # Create result dictionary
    result = {}

    # Get all keys from all items
    all_keys = set()
    for item in batch:
        all_keys.update(item.keys())

    # Process each key
    for key in all_keys:
        if key in ['image', 'input_ids', 'attention_mask']:
            # For tensor data
            try:
                values = [item[key] for item in batch if key in item and item[key] is not None]
                if values:
                    result[key] = torch.stack(values)
            except:
                # Fall back to empty tensors if stacking fails
                pass
        else:
            # For non-tensor data
            result[key] = [item.get(key, None) for item in batch]

    return result

@dataclass
class TrainingConfig:
    """Training configuration to promote creative divergence"""
    run_id: str = None
    # Training parameters
    learning_rate: float = 8e-4
    batch_size: int = 4
    gradient_accumulation_steps: int = 4
    num_epochs: int = 15
    max_grad_norm: float = 1.0

    # Mixed precision and optimization
    use_mixed_precision: bool = True
    optimizer: str = "AdamW"

    # LR scheduling
    lr_scheduler: str = "cosine"
    lr_warmup_steps: int = 50

    # Validation and checkpointing
    validation_split: float = 0.1
    val_check_interval: int = 100
    checkpoint_interval: int = 200
    patience: int = 10

    # LoRA parameters - significantly increased
    lora_rank: int = 64  # High rank for more expressive capacity
    lora_alpha: float = 128.0  # High scaling for stronger effect

    # Audio-specific parameters
    audio_sample_rate: int = 44100
    max_token_length: int = 77

    # Paths
    output_dir: str = os.path.join(OUTPUT_PATH, "finetuned")


class SpectrogramDataset(Dataset):
    """Dataset for training with audio spectrograms"""
    def __init__(
        self,
        training_data,
        tokenizer,
        config,
        split="train",
        validation_ratio=0.1,
        seed=42
    ):
        self.tokenizer = tokenizer
        self.max_token_length = config.max_token_length
        self.audio_sample_rate = config.audio_sample_rate

        # Pre-filter dataset to remove samples with missing files
        filtered_data = []
        for item in training_data:
            image_path = item.get('image_path', '')
            if image_path and os.path.exists(image_path):
                filtered_data.append(item)

        print(f"Filtered dataset to {len(filtered_data)} valid samples")
        training_data = filtered_data

        if not training_data:
            raise ValueError("No valid training samples found with existing image files")

        # Split data into train/validation
        np.random.seed(seed)
        all_indices = np.arange(len(training_data))
        np.random.shuffle(all_indices)

        val_size = int(len(training_data) * validation_ratio)
        val_indices = all_indices[:val_size]
        train_indices = all_indices[val_size:]

        # Assign data based on split
        if split == "train":
            self.data_indices = train_indices
        elif split == "val":
            self.data_indices = val_indices
        else:
            raise ValueError(f"Split {split} not recognized. Use 'train' or 'val'.")

        self.training_data = training_data

        # Augmentation and normalization transforms
        if split == "train":
            self.transform = transforms.Compose([
                transforms.RandomHorizontalFlip(p=0.05),  # Minimal time reversal
                transforms.RandomAffine(
                    degrees=0, translate=(0.02, 0.02),
                    scale=(0.98, 1.02), fill=0),
                # No color jitter to preserve spectral characteristics
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ])
        else:
            # No augmentation for validation
            self.transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ])

    def __len__(self):
        return len(self.data_indices)

    def __getitem__(self, idx):
        try:
            true_idx = self.data_indices[idx]
            item = self.training_data[true_idx]

            # Load image and verify it exists
            image_path = item.get('image_path', '')
            if not image_path or not os.path.exists(image_path):
                print(f"Warning: Image file not found: {image_path}")
                return None

            # Load and transform image
            try:
                image = Image.open(image_path).convert('RGB')
            except Exception as e:
                print(f"Error opening image {image_path}: {e}")
                return None

            # Resize to expected dimensions if needed
            if image.height != 512 or image.width < 100:
                image = image.resize((512, 512), Image.LANCZOS)

            # Apply transforms (including augmentations for training)
            try:
                image = self.transform(image)
                # Make sure image is float32 to match model loading dtype
                image = image.float()
            except Exception as e:
                print(f"Error transforming image {image_path}: {e}")
                return None

            # Prompt construction with domain-specific attributes
            base_prompt = item.get('prompt', 'audio sample')
            metadata = item.get('metadata', {})
            enhanced_prompt = base_prompt

            # Add musical metadata with variations
            musical_attributes = []

            # Add clip position information if available
            if 'clip_index' in metadata and 'total_clips' in metadata:
                clip_num = metadata['clip_index']
                total_clips = metadata['total_clips']

                # Occasionally add section information to prompt (30% chance)
                if random.random() < 0.3:
                    if clip_num == 1:
                        musical_attributes.append("intro section")
                    elif clip_num == total_clips:
                        musical_attributes.append("outro section")
                    elif clip_num < total_clips / 3:
                        musical_attributes.append("early section")
                    elif clip_num > 2 * total_clips / 3:
                        musical_attributes.append("late section")
                    else:
                        musical_attributes.append("middle section")

            # Add tempo/BPM with slight variations
            bpm = metadata.get('bpm')
            if bpm and bpm > 0:
                # For training, slightly vary the BPM to prevent overfitting
                if random.random() < 0.3:  # 30% chance of slight variation
                    bpm_variation = bpm + random.randint(-5, 5)
                    musical_attributes.append(f"{bpm_variation} BPM")
                else:
                    musical_attributes.append(f"{bpm} BPM")

            # Add key information
            key = metadata.get('key')
            if key and key != "unknown":
                musical_attributes.append(f"key of {key}")

            # Add genre with occasional rephrasing
            genre = metadata.get('genre')
            if genre and genre != "unknown":
                genre_phrases = [
                    f"{genre} style",
                    f"{genre} music",
                    f"{genre} genre",
                    f"in {genre} style"
                ]
                musical_attributes.append(random.choice(genre_phrases))

            # Add file type descriptor for more specific generation
            file_type = metadata.get('file_type')
            if file_type and file_type != "unknown":
                type_descriptions = {
                    "drum": ["drums", "percussion", "rhythm", "beat"],
                    "melodic": ["melodic", "melody", "harmonic", "musical"],
                    "fx": ["effect", "sound effect", "atmospheric", "textural"],
                    "vocal": ["vocal", "voice", "singing", "spoken"]
                }

                if file_type in type_descriptions:
                    descriptions = type_descriptions[file_type]
                    musical_attributes.append(random.choice(descriptions))

            # Add musical attributes to prompt in randomized order to increase variety
            random.shuffle(musical_attributes)
            if musical_attributes:
                enhanced_prompt += ", " + ", ".join(musical_attributes)

            # Occasionally add production quality descriptors for variety
            if random.random() < 0.3:
                quality_descriptors = [
                    "high quality", "professional", "clean", "punchy",
                    "warm", "crisp", "dynamic", "balanced"
                ]
                enhanced_prompt += f", {random.choice(quality_descriptors)}"

            # Tokenize prompt
            try:
                tokens = self.tokenizer(
                    enhanced_prompt,
                    padding="max_length",
                    max_length=self.max_token_length,
                    truncation=True,
                    return_tensors="pt"
                )
                input_ids = tokens.input_ids[0]
                attention_mask = tokens.attention_mask[0]
            except Exception as e:
                print(f"Error tokenizing prompt '{enhanced_prompt}': {e}")
                # Provide default token values for this case
                input_ids = torch.zeros(self.max_token_length, dtype=torch.long)
                attention_mask = torch.zeros(self.max_token_length, dtype=torch.long)

            return {
                "image": image,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "original_prompt": base_prompt,
                "enhanced_prompt": enhanced_prompt,
                "metadata": metadata
            }
        except Exception as e:
            print(f"Unexpected error in dataset __getitem__: {e}")
            return None



class WeightedSpectrogramDataset(SpectrogramDataset):
    """Dataset with built-in source file weighting"""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Compute weights after dataset splitting happens in parent class
        self.weights = self._compute_weights()

    def _compute_weights(self):
        """Compute weights inverse to source file frequency"""
        # Only consider items in this dataset split
        data_indices = self.data_indices
        # Count source files only in this split
        source_counts = {}
        for idx in data_indices:
            item = self.training_data[idx]
            metadata = item.get('metadata', {})
            source_file = metadata.get('source_file', metadata.get('path', 'unknown'))
            source_counts[source_file] = source_counts.get(source_file, 0) + 1

        # Calculate weights for just this split
        weights = []
        for idx in data_indices:
            item = self.training_data[idx]
            metadata = item.get('metadata', {})
            source_file = metadata.get('source_file', metadata.get('path', 'unknown'))
            count = max(1, source_counts.get(source_file, 1))
            weights.append(1.0 / count)

        return weights

    def get_weighted_sampler(self):
        """Return a weighted sampler for this dataset"""
        # Create a sampler for indices only within this dataset split
        return torch.utils.data.WeightedRandomSampler(
            weights=self.weights,
            num_samples=len(self.weights),
            replacement=True
        )

### 5.2 LoRA Setup and Training Functions

In [12]:
from torch.cuda.amp import autocast, GradScaler
from diffusers.optimization import get_scheduler
import math
import copy

def set_up_custom_lora(unet, rank=32, alpha=128.0):
    """
    Enhanced LoRA setup with significantly more trainable parameters
    targeting ~3-5% of total model capacity.

    Args:
        unet: UNet model from diffusers
        rank: Rank of low-rank decomposition (increased)
        alpha: Scaling factor for LoRA (increased)

    Returns:
        Modified UNet, trainable parameters, and original state for restoration
    """
    # Get original parameters for later comparison
    original_state = copy.deepcopy(unet.state_dict())

    # Start by freezing all parameters
    for param in unet.parameters():
        param.requires_grad = False

    trainable_params = []
    adapted_layers = []

    # ===== Target ALL potential layers in the model =====

    # 1. Get all attention blocks throughout the model
    attention_modules = []

    # Process mid block attentions
    if hasattr(unet.mid_block, 'attentions'):
        attention_modules.extend(unet.mid_block.attentions)

    # Process up blocks attentions
    for up_block in unet.up_blocks:
        if hasattr(up_block, 'attentions'):
            attention_modules.extend(up_block.attentions)

    # Process down blocks attentions
    for down_block in unet.down_blocks:
        if hasattr(down_block, 'attentions'):
            attention_modules.extend(down_block.attentions)

    # 2. Get all resnet blocks throughout the model
    resnet_modules = []

    # Mid block resnets
    if hasattr(unet.mid_block, 'resnets'):
        resnet_modules.extend(unet.mid_block.resnets)

    # Up block resnets
    for up_block in unet.up_blocks:
        if hasattr(up_block, 'resnets'):
            resnet_modules.extend(up_block.resnets)

    # Down block resnets
    for down_block in unet.down_blocks:
        if hasattr(down_block, 'resnets'):
            resnet_modules.extend(down_block.resnets)

    # 3. Apply LoRA to all attention modules
    for attn_module in attention_modules:
        for transformer in attn_module.transformer_blocks:
            # Self-attention
            if hasattr(transformer, 'attn1'):
                for proj in [transformer.attn1.to_q, transformer.attn1.to_k,
                            transformer.attn1.to_v, transformer.attn1.to_out[0]]:
                    create_lora_adapter(proj, rank, alpha)
                    for name, param in proj.named_parameters():
                        if param.requires_grad:
                            trainable_params.append(param)
                    adapted_layers.append(proj)

            # Cross-attention
            if hasattr(transformer, 'attn2'):
                for proj in [transformer.attn2.to_q, transformer.attn2.to_k,
                            transformer.attn2.to_v, transformer.attn2.to_out[0]]:
                    create_lora_adapter(proj, rank, alpha)
                    for name, param in proj.named_parameters():
                        if param.requires_grad:
                            trainable_params.append(param)
                    adapted_layers.append(proj)

            # Feed-forward networks
            if hasattr(transformer, 'ff'):
                if hasattr(transformer.ff, 'net'):
                    # Some diffusers versions use 'net'
                    for layer in transformer.ff.net:
                        if isinstance(layer, torch.nn.Linear):
                            create_lora_adapter(layer, rank, alpha)
                            for name, param in layer.named_parameters():
                                if param.requires_grad:
                                    trainable_params.append(param)
                            adapted_layers.append(layer)
                # Older versions might use other attribute names
                elif hasattr(transformer.ff, 'proj'):
                    create_lora_adapter(transformer.ff.proj, rank, alpha)
                    for name, param in transformer.ff.proj.named_parameters():
                        if param.requires_grad:
                            trainable_params.append(param)
                    adapted_layers.append(transformer.ff.proj)

    # 4. Apply LoRA to all resnet convolutions
    for resnet in resnet_modules:
        for layer_name in ['conv1', 'conv2', 'conv_shortcut']:
            if hasattr(resnet, layer_name):
                layer = getattr(resnet, layer_name)
                if isinstance(layer, torch.nn.Conv2d):
                    create_lora_adapter(layer, rank, alpha)
                    # Add all trainable parameters from this layer
                    for name, param in layer.named_parameters():
                        if param.requires_grad:
                            trainable_params.append(param)
                    adapted_layers.append(layer)

    # 5. Apply LoRA to the key convolution layers in the model

    # Main output conv (critical for final generation)
    create_lora_adapter(unet.conv_out, rank, alpha)
    for name, param in unet.conv_out.named_parameters():
        if param.requires_grad:
            trainable_params.append(param)
    adapted_layers.append(unet.conv_out)

    # Input convolution
    if hasattr(unet, 'conv_in'):
        create_lora_adapter(unet.conv_in, rank, alpha)
        for name, param in unet.conv_in.named_parameters():
            if param.requires_grad:
                trainable_params.append(param)
        adapted_layers.append(unet.conv_in)

    # Count trainable parameters
    trainable_count = sum(p.numel() for p in trainable_params)
    total_count = sum(p.numel() for p in unet.parameters())
    print(f"Added LoRA adapters to {len(adapted_layers)} layers")
    print(f"Trainable parameters: {trainable_count:,} ({trainable_count/total_count:.2%} of total)")

    return unet, trainable_params, original_state

def create_lora_adapter(module, name, rank, alpha):
    """Create a LoRA adapter for a module with proper dimension handling."""
    import types
    import math

    # Store original forward
    original_forward = module.forward

    # Initialize LoRA for Linear layers
    if isinstance(module, torch.nn.Linear):
        in_features = module.in_features
        out_features = module.out_features

        # Create A and B matrices
        module.lora_A = torch.nn.Parameter(torch.zeros(in_features, rank))
        module.lora_B = torch.nn.Parameter(torch.zeros(rank, out_features))

        # Initialize weights
        torch.nn.init.kaiming_uniform_(module.lora_A, a=math.sqrt(5))
        torch.nn.init.zeros_(module.lora_B)

        # Set scaling factor
        module.lora_scaling = alpha / rank

        # Enable gradient computation
        module.lora_A.requires_grad_(True)
        module.lora_B.requires_grad_(True)

        # Define new forward method
        def lora_linear_forward(self, x):
            original_output = original_forward(x)
            lora_output = (x @ self.lora_A) @ self.lora_B
            return original_output + lora_output * self.lora_scaling

        # Replace forward method
        module.forward = types.MethodType(lora_linear_forward, module)

    # Initialize LoRA for Conv2d layers
    elif isinstance(module, torch.nn.Conv2d):
        in_channels = module.in_channels
        out_channels = module.out_channels

        # Create LoRA as separate Conv2d modules with 1x1 kernels
        module.lora_A = torch.nn.Conv2d(
            in_channels=in_channels,
            out_channels=rank,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )

        module.lora_B = torch.nn.Conv2d(
            in_channels=rank,
            out_channels=out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=False
        )

        # Initialize weights
        torch.nn.init.kaiming_uniform_(module.lora_A.weight, a=math.sqrt(5))
        torch.nn.init.zeros_(module.lora_B.weight)

        # Set scaling factor
        module.lora_scaling = alpha / rank

        # Enable gradient computation
        for param in module.lora_A.parameters():
            param.requires_grad = True
        for param in module.lora_B.parameters():
            param.requires_grad = True

        # Define new forward method with explicit shape handling
        def lora_conv_forward(self, x):
            original_output = original_forward(x)

            # Process through LoRA path with dimension checks
            lora_a_output = self.lora_A(x)
            lora_b_output = self.lora_B(lora_a_output)

            # Ensure output shape matches original output
            if lora_b_output.shape != original_output.shape:
                # Resize to match original output's spatial dimensions
                lora_b_output = torch.nn.functional.interpolate(
                    lora_b_output,
                    size=original_output.shape[2:],
                    mode='bilinear',
                    align_corners=False
                )

            return original_output + lora_b_output * self.lora_scaling

        # Replace forward method
        module.forward = types.MethodType(lora_conv_forward, module)

    # Store name for debugging
    module.lora_name = name

    return module

def train_model(
    pipeline,
    train_dataset,
    val_dataset=None,
    config=None,
):
    """
    Improved training function with better learning dynamics and data utilization.
    Ensures consistent dtype handling throughout the pipeline.
    """
    if config is None:
        config = TrainingConfig()

    # Create output directory
    os.makedirs(config.output_dir, exist_ok=True)

    # Set up device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Create data loaders with custom collate function
    train_dataloader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        collate_fn=custom_collate_fn,
        drop_last=True
    )

    if val_dataset:
        val_dataloader = DataLoader(
            val_dataset,
            batch_size=config.batch_size,
            shuffle=False,
            num_workers=2,
            pin_memory=True,
            collate_fn=custom_collate_fn,
            drop_last=True
        )
    else:
        val_dataloader = None

    # Set up UNet for training using improved LoRA approach
    unet = pipeline.unet
    unet, trainable_params, original_state = set_up_custom_lora(
        unet, rank=config.lora_rank, alpha=config.lora_alpha
    )
    unet.to(device)

    # Ensure UNet dtype matches VAE and text encoder
    model_dtype = next(pipeline.vae.parameters()).dtype
    print(f"Model dtype: {model_dtype}")

    # Set up optimizer with higher learning rate
    # Remove duplicate parameters to avoid warning
    unique_params_dict = {}
    for param in trainable_params:
        unique_params_dict[param.data_ptr()] = param
    unique_params = list(unique_params_dict.values())
    optimizer = torch.optim.AdamW(
        unique_params,  # Use unique params
        lr=config.learning_rate,
        betas=(0.9, 0.999),
        weight_decay=0.01,
        eps=1e-8,
    )

    # Set up learning rate scheduler with warmup
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / config.gradient_accumulation_steps)
    max_train_steps = config.num_epochs * num_update_steps_per_epoch

    # Create improved scheduler
    if config.lr_scheduler == "cosine_with_restarts":
        from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

        # Convert steps to epochs for CosineAnnealingWarmRestarts
        lr_scheduler = CosineAnnealingWarmRestarts(
            optimizer,
            T_0=3,                  # Restart every 3 epochs
            T_mult=2,               # Double period after each restart
            eta_min=config.learning_rate/10  # Min LR = 10% of max
        )

        # Create a wrapper function to match the expected interface
        def lr_lambda(current_step):
            # Warmup for the first config.lr_warmup_steps
            if current_step < config.lr_warmup_steps:
                return float(current_step) / float(max(1, config.lr_warmup_steps))
            # After warmup, use the scheduler
            return 1.0  # The actual scheduling happens in scheduler.step()

        # Original scheduler just for warmup
        warmup_scheduler = get_scheduler(
            name="constant_with_warmup",
            optimizer=optimizer,
            num_warmup_steps=config.lr_warmup_steps,
            num_training_steps=max_train_steps
        )
    else:
        # Original scheduler if not using cosine_with_restarts
        lr_scheduler = get_scheduler(
            name=config.lr_scheduler,
            optimizer=optimizer,
            num_warmup_steps=config.lr_warmup_steps,
            num_training_steps=max_train_steps
        )

    # Setup for mixed precision training - enable for efficiency
    use_mixed_precision = config.use_mixed_precision and torch.cuda.is_available()
    scaler = GradScaler() if use_mixed_precision else None

    # Training state variables
    global_step = 0
    best_val_loss = float('inf')
    patience_counter = 0

    # Start training
    print(f"Starting training for {config.num_epochs} epochs...")
    start_time = time.time()

    # Save initial config
    with open(os.path.join(config.output_dir, "config.json"), "w") as f:
        config_dict = {k: v for k, v in config.__dict__.items()}
        json.dump(config_dict, f, indent=2)

    # Track loss history for visualization
    train_losses = []
    val_losses = []
    val_steps = []  # Track steps where validation occurred

    for epoch in range(config.num_epochs):
        # Training loop
        unet.train()
        train_loss = 0.0

        progress_bar = tqdm(total=len(train_dataloader), desc=f"Epoch {epoch+1}/{config.num_epochs}")

        for step, batch in enumerate(train_dataloader):
            # Skip empty batches
            if batch is None or len(batch) == 0:
                print("Skipping empty batch")
                continue

            # Move data to device and convert to model dtype
            try:
                pixel_values = batch["image"].to(device, dtype=model_dtype)
                input_ids = batch["input_ids"].to(device)
            except Exception as e:
                print(f"Error processing batch: {e}")
                continue

            # Get text embeddings for conditioning
            with torch.no_grad():
                text_embeddings = pipeline.text_encoder(input_ids)[0]

            # Convert images to latent space
            with torch.no_grad():
                # Encode images - ensure pixel_values matches VAE dtype
                latents = pipeline.vae.encode(pixel_values).latent_dist.sample()
                latents = latents * 0.18215

            # Sample noise
            noise = torch.randn_like(latents)
            batch_size = latents.shape[0]

            # Sample timesteps uniformly across the full range
            timesteps = torch.randint(
                0, pipeline.scheduler.config.num_train_timesteps,
                (batch_size,), device=latents.device
            ).long()

            # Add noise to latents
            noisy_latents = pipeline.scheduler.add_noise(latents, noise, timesteps)

            # Ensure consistent dtype for all tensors
            noisy_latents = noisy_latents.to(dtype=model_dtype)
            timesteps = timesteps.to(device)
            text_embeddings = text_embeddings.to(dtype=model_dtype)
            noise = noise.to(dtype=model_dtype)

            # Predict the noise residual with unet (forward pass)
            with autocast(enabled=use_mixed_precision):
                noise_pred = unet(noisy_latents, timesteps, text_embeddings).sample

                # Calculate loss - use MSE loss
                loss = torch.nn.functional.mse_loss(noise_pred, noise, reduction="mean")
                loss = loss / config.gradient_accumulation_steps

            # Backpropagation with mixed precision
            if use_mixed_precision:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            # Update weights if we've accumulated enough gradients
            if (step + 1) % config.gradient_accumulation_steps == 0:
                if use_mixed_precision:
                    scaler.unscale_(optimizer)

                # Clip gradients
                torch.nn.utils.clip_grad_norm_(trainable_params, config.max_grad_norm)

                # Update weights
                if use_mixed_precision:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                lr_scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                # Track loss
                train_losses.append(loss.item() * config.gradient_accumulation_steps)

                # Save checkpoint periodically
                if global_step % config.checkpoint_interval == 0:
                    checkpoint_dir = os.path.join(config.output_dir, f"checkpoint-{global_step}")
                    os.makedirs(checkpoint_dir, exist_ok=True)

                    # Save model with adapters
                    torch.save(unet.state_dict(), os.path.join(checkpoint_dir, "unet.pt"))
                    print(f"Saved checkpoint at step {global_step} to {checkpoint_dir}")

            # Update progress bar
            train_loss += loss.item() * config.gradient_accumulation_steps
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
            progress_bar.update(1)

            # Validate periodically
            if val_dataloader is not None and global_step % config.val_check_interval == 0:
                val_loss = validate(unet, pipeline, val_dataloader, device, model_dtype, use_mixed_precision)
                print(f"Validation loss: {val_loss:.4f}")
                val_losses.append(val_loss)
                val_steps.append(global_step)  # Store the step number for plotting

                # Early stopping
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    patience_counter = 0

                    # Save best model
                    best_model_dir = os.path.join(config.output_dir, "best_model")
                    os.makedirs(best_model_dir, exist_ok=True)
                    torch.save(unet.state_dict(), os.path.join(best_model_dir, "unet.pt"))
                    print(f"Saved best model with validation loss {val_loss:.4f}")
                else:
                    patience_counter += 1
                    if patience_counter >= config.patience:
                        print(f"Early stopping triggered after {global_step} steps")
                        break

        # End of epoch
        avg_train_loss = train_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{config.num_epochs} complete. Avg loss: {avg_train_loss:.4f}")

        # Early stopping check
        if patience_counter >= config.patience:
            print("Early stopping triggered due to no improvement in validation loss")
            break

    # End of training
    training_time = time.time() - start_time
    print(f"Training completed in {training_time/60:.2f} minutes")

    # Save final model
    final_dir = os.path.join(config.output_dir, "final_model")
    os.makedirs(final_dir, exist_ok=True)
    torch.save(unet.state_dict(), os.path.join(final_dir, "unet.pt"))

    # Save training losses for visualization
    np.savetxt(os.path.join(config.output_dir, 'training_loss.txt'), np.array(train_losses))
    if val_losses:
        np.savetxt(os.path.join(config.output_dir, 'validation_loss.txt'), np.array(val_losses))
        np.savetxt(os.path.join(config.output_dir, 'validation_steps.txt'), np.array(val_steps))

    # Visualize training progress - FIX: properly handling validation steps
    plt.figure(figsize=(10, 5))
    plt.plot(range(len(train_losses)), train_losses, label='Training Loss')

    if val_losses and val_steps:
        # Use the actual validation steps instead of assumption
        plt.plot(val_steps[:len(val_losses)], val_losses, 'o-', label='Validation Loss')

    plt.xlabel('Steps')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training Progress')
    plt.savefig(os.path.join(config.output_dir, 'training_progress.png'))
    plt.close()

    return final_dir

def validate_audio_quality(unet, pipeline, val_dataloader, device, model_dtype, use_mixed_precision=False):
    """Validate with focus on both reconstruction loss and audio quality metrics"""
    unet.eval()
    val_loss = 0.0
    spectral_contrast_loss = 0.0
    batch_count = 0

    # Track validation statistics
    total_batches = 0
    empty_batches = 0
    malformed_batches = 0
    shape_errors = 0
    processing_errors = 0

    print(f"Starting validation on {len(val_dataloader)} batches...")

    with torch.no_grad():
        for batch_idx, batch in enumerate(val_dataloader):
            total_batches += 1

            # Skip invalid batches
            if batch is None:
                print(f"Batch {batch_idx}: Empty batch (None)")
                empty_batches += 1
                continue

            # Check batch content
            if not isinstance(batch, dict) or 'image' not in batch or 'input_ids' not in batch:
                print(f"Batch {batch_idx}: Malformed batch structure")
                malformed_batches += 1
                continue

            # Check tensor shapes
            if batch['image'].shape[0] == 0 or batch['input_ids'].shape[0] == 0:
                print(f"Batch {batch_idx}: Empty tensors in batch")
                shape_errors += 1
                continue

            try:
                # Process batch
                pixel_values = batch["image"].to(device, dtype=model_dtype)
                input_ids = batch["input_ids"].to(device)

                # Get text embeddings
                text_embeddings = pipeline.text_encoder(input_ids)[0]
                text_embeddings = text_embeddings.to(dtype=model_dtype)

                # Encode to latent space
                latents = pipeline.vae.encode(pixel_values).latent_dist.sample()
                latents = latents * 0.18215

                # Sample noise
                noise = torch.randn_like(latents)
                batch_size = latents.shape[0]

                # Sample timesteps (single step for validation)
                timestep = torch.randint(
                    10, pipeline.scheduler.config.num_train_timesteps - 10,
                    (1,), device=device
                ).long()

                # Add noise to latents with a manually calculated approach instead of using scheduler
                # This is more compatible across scheduler types
                alphas = pipeline.scheduler.alphas_cumprod
                alpha_t = alphas[timestep].item()
                sqrt_alpha_t = np.sqrt(alpha_t)
                sqrt_one_minus_alpha_t = np.sqrt(1 - alpha_t)

                noisy_latents = sqrt_alpha_t * latents + sqrt_one_minus_alpha_t * noise

                # Predict noise with UNet
                with torch.cuda.amp.autocast(enabled=use_mixed_precision):
                    noise_pred = unet(noisy_latents, timestep, text_embeddings).sample

                    # Basic MSE loss
                    mse_loss = torch.nn.functional.mse_loss(noise_pred, noise, reduction="mean")

                    # IMPORTANT: For latent reconstruction, use a simpler approach
                    # instead of pipeline.scheduler.step which is causing the error
                    # This is a simpler denoising step that works across scheduler types
                    pred_original_sample = (noisy_latents - sqrt_one_minus_alpha_t * noise_pred) / sqrt_alpha_t

                    # Calculate reconstruction quality
                    latent_diff = torch.nn.functional.mse_loss(pred_original_sample, latents)

                val_loss += mse_loss.item()
                spectral_contrast_loss += latent_diff.item()
                batch_count += 1

            except Exception as e:
                print(f"Batch {batch_idx}: Error processing validation batch: {e}")
                processing_errors += 1
                continue

    print(f"\nValidation Statistics:")
    print(f"  Total batches: {total_batches}")
    print(f"  Successfully processed: {batch_count} ({batch_count/max(1, total_batches):.1%})")
    print(f"  Empty batches: {empty_batches}")
    print(f"  Malformed batches: {malformed_batches}")
    print(f"  Shape errors: {shape_errors}")
    print(f"  Processing errors: {processing_errors}")

    # Average losses
    if batch_count > 0:
        avg_val_loss = val_loss / batch_count
        avg_spectral_loss = spectral_contrast_loss / batch_count
    else:
        print("WARNING: No valid validation batches processed!")
        avg_val_loss = 0.0
        avg_spectral_loss = 0.0

    unet.train()
    return avg_val_loss, avg_spectral_loss


def validate(unet, pipeline, val_dataloader, device, model_dtype, use_mixed_precision=False):
    """Validate the model on the validation dataset"""
    unet.eval()
    val_loss = 0.0
    batch_count = 0

    with torch.no_grad():
        for batch in val_dataloader:
            # Skip empty batches
            if batch is None or len(batch) == 0:
                continue

            try:
                # Move data to device and match dtype
                pixel_values = batch["image"].to(device, dtype=model_dtype)
                input_ids = batch["input_ids"].to(device)

                # Get text embeddings
                text_embeddings = pipeline.text_encoder(input_ids)[0]
                text_embeddings = text_embeddings.to(dtype=model_dtype)

                # Convert images to latent space
                latents = pipeline.vae.encode(pixel_values).latent_dist.sample()
                latents = latents * 0.18215

                # Sample noise
                noise = torch.randn_like(latents)
                batch_size = latents.shape[0]

                # Sample timesteps
                timesteps = torch.randint(
                    0, pipeline.scheduler.config.num_train_timesteps,
                    (batch_size,), device=latents.device
                ).long()

                # Add noise to latents
                noisy_latents = pipeline.scheduler.add_noise(latents, noise, timesteps)

                # Ensure all tensors have correct dtype
                noisy_latents = noisy_latents.to(dtype=model_dtype)
                timesteps = timesteps.to(device)
                noise = noise.to(dtype=model_dtype)

                # Predict the noise residual
                with autocast(enabled=use_mixed_precision):
                    noise_pred = unet(noisy_latents, timesteps, text_embeddings).sample
                    loss = torch.nn.functional.mse_loss(noise_pred, noise, reduction="mean")

                val_loss += loss.item()
                batch_count += 1
            except Exception as e:
                print(f"Error in validation batch: {e}")
                continue

    # Average loss
    avg_val_loss = val_loss / max(batch_count, 1)
    unet.train()  # Set back to training mode

    return avg_val_loss

### 5.3 Execute Fine-tuning Process

Final fully optimized training loop

In [ ]:
from transformers import get_scheduler

def optimized_training():
    """Training approach with optimized parameter coverage (7-10%)"""
    # Output path
    output_path = '/content/drive/MyDrive/riffusion_finetuning'
    training_data_file = os.path.join(output_path, "training_data.json")

    # Create timestamp
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    output_dir = os.path.join(output_path, f"optimized_{timestamp}")
    os.makedirs(output_dir, exist_ok=True)

    # Load training data
    print("Loading training data...")
    with open(training_data_file, 'r') as f:
        training_data = json.load(f)
    print(f"Loaded {len(training_data)} samples")

    # Load model
    print("Loading base Riffusion model...")
    from diffusers import DiffusionPipeline
    device = "cuda" if torch.cuda.is_available() else "cpu"
    pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None,
        requires_safety_checker=False
    )
    pipeline.to(device)
    print(f"Model loaded and moved to {device}")
    # Create dataset config with improved parameters based on previous runs
    config = TrainingConfig(
        run_id=f"optimized_{timestamp}",
        learning_rate=5e-5,
        batch_size=4,
        gradient_accumulation_steps=4,
        num_epochs=30,                 # Increased from 20 after decreasing learing rate
        max_grad_norm=0.5,
        use_mixed_precision=True,
        lr_scheduler="cosine",
        lr_warmup_steps=200,           # Reduced from 300
        validation_split=0.1,
        val_check_interval=50,
        patience=8,                    # Added patience parameter for early stopping
        # weight_decay=0.01,           # Added weight decay then later removed
        output_dir=output_dir
    )
    # Create datasets with separate weighting
    print("Creating weighted training dataset...")
    weighted_train_dataset = WeightedSpectrogramDataset(
        training_data=training_data,
        tokenizer=pipeline.tokenizer,
        config=config,
        split="train",
        validation_ratio=config.validation_split,
        seed=42
    )
    print("Creating validation dataset...")
    validation_dataset = SpectrogramDataset(
        training_data=training_data,
        tokenizer=pipeline.tokenizer,
        config=config,
        split="val",
        validation_ratio=config.validation_split,
        seed=42
    )
    print(f"Training dataset size: {len(weighted_train_dataset)}")
    print(f"Validation dataset size: {len(validation_dataset)}")

    # Set up selective fine-tuning (best config found for best quality)
    print("Setting up fine-tuning parameters...")
    trainable_params = []
    total_params = sum(p.numel() for p in pipeline.unet.parameters())

    # Freeze all parameters first
    for param in pipeline.unet.parameters():
        param.requires_grad = False
    # 1. Full mid-block attention components
    attn_count = 0
    if hasattr(pipeline.unet.mid_block, 'attentions'):
        for attn in pipeline.unet.mid_block.attentions:
            if hasattr(attn, 'transformer_blocks'):
                for block in attn.transformer_blocks:
                    # Include ALL projections in self-attention
                    if hasattr(block, 'attn1'):
                        for proj_name in ['to_q', 'to_k', 'to_v', 'to_out']:
                            if hasattr(block.attn1, proj_name):
                                if proj_name == 'to_out':
                                    # Handle to_out which is usually a list
                                    if isinstance(block.attn1.to_out, list) and len(block.attn1.to_out) > 0:
                                        block.attn1.to_out[0].requires_grad_(True)
                                        trainable_params.extend(block.attn1.to_out[0].parameters())
                                        attn_count += 1
                                else:
                                    # Regular projection
                                    getattr(block.attn1, proj_name).requires_grad_(True)
                                    trainable_params.extend(getattr(block.attn1, proj_name).parameters())
                                    attn_count += 1

                    # Add cross-attention components
                    if hasattr(block, 'attn2'):
                        for proj_name in ['to_q', 'to_k', 'to_v', 'to_out']:
                            if hasattr(block.attn2, proj_name):
                                if proj_name == 'to_out':
                                    # Handle to_out which is a list
                                    if isinstance(block.attn2.to_out, list) and len(block.attn2.to_out) > 0:
                                        block.attn2.to_out[0].requires_grad_(True)
                                        trainable_params.extend(block.attn2.to_out[0].parameters())
                                        attn_count += 1
                                else:
                                    # Regular projection
                                    getattr(block.attn2, proj_name).requires_grad_(True)
                                    trainable_params.extend(getattr(block.attn2, proj_name).parameters())
                                    attn_count += 1
    print(f"Added {attn_count} mid-block attention projections")
    # 2. Up-blocks for audio detail (3 blocks)
    up_attn_count = 0
    up_blocks_to_process = min(3, len(pipeline.unet.up_blocks))
    for i in range(len(pipeline.unet.up_blocks) - up_blocks_to_process, len(pipeline.unet.up_blocks)):
        if hasattr(pipeline.unet.up_blocks[i], 'attentions'):
            for attn_idx, attn in enumerate(pipeline.unet.up_blocks[i].attentions):
                if hasattr(attn, 'transformer_blocks'):
                    # Process ALL transformer blocks
                    for block_idx, block in enumerate(attn.transformer_blocks):
                        if hasattr(block, 'attn1'):
                            # Add ALL projections for better coverage
                            for proj_name in ['to_q', 'to_k', 'to_v', 'to_out']:
                                if hasattr(block.attn1, proj_name):
                                    if proj_name == 'to_out':
                                        if isinstance(block.attn1.to_out, list) and len(block.attn1.to_out) > 0:
                                            block.attn1.to_out[0].requires_grad_(True)
                                            trainable_params.extend(block.attn1.to_out[0].parameters())
                                            up_attn_count += 1
                                    else:
                                        getattr(block.attn1, proj_name).requires_grad_(True)
                                        trainable_params.extend(getattr(block.attn1, proj_name).parameters())
                                        up_attn_count += 1
    # 3. Down-blocks (3 blocks)
    down_attn_count = 0
    down_blocks_to_process = min(3, len(pipeline.unet.down_blocks))
    for i in range(len(pipeline.unet.down_blocks) - down_blocks_to_process, len(pipeline.unet.down_blocks)):
        down_block = pipeline.unet.down_blocks[i]
        if hasattr(down_block, 'attentions'):
            for attn in down_block.attentions:
                if hasattr(attn, 'transformer_blocks'):
                    for block in attn.transformer_blocks:
                        if hasattr(block, 'attn1'):
                            # Add more projections for each block
                            for proj_name in ['to_q', 'to_k', 'to_v', 'to_out']:
                                if hasattr(block.attn1, proj_name):
                                    if proj_name == 'to_out':
                                        if isinstance(block.attn1.to_out, list) and len(block.attn1.to_out) > 0:
                                            block.attn1.to_out[0].requires_grad_(True)
                                            trainable_params.extend(block.attn1.to_out[0].parameters())
                                            down_attn_count += 1
                                    else:
                                        getattr(block.attn1, proj_name).requires_grad_(True)
                                        trainable_params.extend(getattr(block.attn1, proj_name).parameters())
                                        down_attn_count += 1
    print(f"Added {down_attn_count} down-block attention projections")
    print(f"Added {up_attn_count} up-block attention projections")
    # 4. Cross-attention for prompt control
    cross_attn_count = 0
    max_cross_attns = 12  # Increased from 4 to 12
    # Process blocks from ALL up_blocks
    for i in range(len(pipeline.unet.up_blocks)):
        if hasattr(pipeline.unet.up_blocks[i], 'attentions') and cross_attn_count < max_cross_attns:
            for attn in pipeline.unet.up_blocks[i].attentions:
                if hasattr(attn, 'transformer_blocks') and cross_attn_count < max_cross_attns:
                    for block in attn.transformer_blocks:
                        if hasattr(block, 'attn2') and cross_attn_count < max_cross_attns:  # Cross-attention
                            # Add BOTH to_k and to_v for better control
                            block.attn2.to_k.requires_grad_(True)
                            trainable_params.extend(block.attn2.to_k.parameters())
                            block.attn2.to_v.requires_grad_(True)
                            trainable_params.extend(block.attn2.to_v.parameters())
                            cross_attn_count += 1

    print(f"Added {cross_attn_count} cross-attention components")


    """The following has been commented out to REDUCE the percentage of the UNet being Retrained from ~30% --> ~8%"""
    # # 5. Add ResNet components (important for spatial features)
    # resnet_count = 0
    # max_resnets = 12  # Add up to 12 ResNets

    # # Add ResNets from both up and down blocks
    # for block_group in [pipeline.unet.up_blocks, pipeline.unet.down_blocks]:
    #     for block_idx, block in enumerate(block_group):
    #         if hasattr(block, 'resnets') and resnet_count < max_resnets:
    #             for resnet_idx, resnet in enumerate(block.resnets):
    #                 if resnet_count < max_resnets:
    #                     # Add both conv1 and conv2
    #                     for layer_name in ['conv1', 'conv2']:
    #                         if hasattr(resnet, layer_name):
    #                             getattr(resnet, layer_name).requires_grad_(True)
    #                             trainable_params.extend(getattr(resnet, layer_name).parameters())
    #                             resnet_count += 1

    # # Also add mid-block resnets
    # if hasattr(pipeline.unet.mid_block, 'resnets') and resnet_count < max_resnets:
    #     for resnet in pipeline.unet.mid_block.resnets:
    #         for layer_name in ['conv1', 'conv2']:
    #             if hasattr(resnet, layer_name) and resnet_count < max_resnets:
    #                 getattr(resnet, layer_name).requires_grad_(True)
    #                 trainable_params.extend(getattr(resnet, layer_name).parameters())
    #                 resnet_count += 1

    # print(f"Added {resnet_count} ResNet components")

    # 6. Add time embeddings
    time_embed_count = 0
    if hasattr(pipeline.unet, 'time_embedding'):
        for layer_name in ['linear_1', 'linear_2']:
            if hasattr(pipeline.unet.time_embedding, layer_name):
                getattr(pipeline.unet.time_embedding, layer_name).requires_grad_(True)
                trainable_params.extend(getattr(pipeline.unet.time_embedding, layer_name).parameters())
                time_embed_count += 1

    print(f"Added {time_embed_count} time embedding layers")

    # 7. Add input projection
    input_count = 0
    if hasattr(pipeline.unet, 'conv_in'):
        pipeline.unet.conv_in.requires_grad_(True)
        trainable_params.extend(pipeline.unet.conv_in.parameters())
        input_count += 1
        print(f"Added input projection with {input_count} layers")

    # 4. Output projection (important for final spectrogram quality)
    pipeline.unet.conv_out.requires_grad_(True)
    trainable_params.extend(pipeline.unet.conv_out.parameters())
    print("Added output projection")

    # 5. Very limited ResNet components (reduced from 15 to 8)
    resnet_count = 0
    max_resnets = 8

    # Only add the final up-block's first ResNet
    if hasattr(pipeline.unet.up_blocks[-1], 'resnets') and resnet_count < max_resnets:
        resnet = pipeline.unet.up_blocks[-1].resnets[0]
        if hasattr(resnet, 'conv2'):
            resnet.conv2.requires_grad_(True)
            trainable_params.extend(resnet.conv2.parameters())
            resnet_count += 1

    # Add one mid-block ResNet if available
    if hasattr(pipeline.unet.mid_block, 'resnets') and resnet_count < max_resnets:
        if pipeline.unet.mid_block.resnets:
            resnet = pipeline.unet.mid_block.resnets[0]
            if hasattr(resnet, 'conv1'):
                resnet.conv1.requires_grad_(True)
                trainable_params.extend(resnet.conv1.parameters())
                resnet_count += 1
    print(f"Added {resnet_count} ResNet components")

    # 6. Add input projection for more parameter coverage
    if hasattr(pipeline.unet, 'conv_in'):
        pipeline.unet.conv_in.requires_grad_(True)
        trainable_params.extend(pipeline.unet.conv_in.parameters())
        print("Added input projection")

    # 7. Add time embeddings
    time_embed_count = 0
    if hasattr(pipeline.unet, 'time_embedding'):
        if hasattr(pipeline.unet.time_embedding, 'linear_1'):
            pipeline.unet.time_embedding.linear_1.requires_grad_(True)
            trainable_params.extend(pipeline.unet.time_embedding.linear_1.parameters())
            time_embed_count += 1
        if hasattr(pipeline.unet.time_embedding, 'linear_2'):
            pipeline.unet.time_embedding.linear_2.requires_grad_(True)
            trainable_params.extend(pipeline.unet.time_embedding.linear_2.parameters())
            time_embed_count += 1
    print(f"Added {time_embed_count} time embedding layers")

    # Track the parameter percentage
    trainable_count = sum(p.numel() for p in trainable_params)
    param_percentage = trainable_count / total_params
    print(f"\nSelectively fine-tuning {len(trainable_params)} parameter groups")
    print(f"Trainable parameters: {trainable_count:,} ({param_percentage:.2%} of UNet)")

    # Get weighted sampler from dataset
    sampler = weighted_train_dataset.get_weighted_sampler()
    # Create dataloader with weighted sampling
    train_dataloader = torch.utils.data.DataLoader(
        weighted_train_dataset,
        batch_size=config.batch_size,
        sampler=sampler,  # Use weighted sampler
        shuffle=False,    # Can't use both sampler and shuffle.
                          # Sampler was found to work best
        collate_fn=custom_collate_fn,
        num_workers=2
    )
    val_dataloader = torch.utils.data.DataLoader(
        validation_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        collate_fn=custom_collate_fn,
        num_workers=2
    )
    # Set up optimizer with weight decay
    optimizer = optim.AdamW(
        trainable_params,
        lr=config.learning_rate,
        betas=(0.9, 0.999),
        weight_decay=0.01,
        eps=1e-8
    )
    # Calculate steps
    num_update_steps_per_epoch = len(train_dataloader)
    max_train_steps = config.num_epochs * num_update_steps_per_epoch
    # Create scheduler
    lr_scheduler = get_scheduler(
        name="cosine",
        optimizer=optimizer,
        num_warmup_steps=config.lr_warmup_steps,
        num_training_steps=max_train_steps
    )
    # Set up mixed precision
    use_mixed_precision = config.use_mixed_precision and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler() if use_mixed_precision else None
    # Get model dtype
    model_dtype = next(pipeline.unet.parameters()).dtype

    # Training loop
    global_step = 0
    best_val_loss = float('inf')
    training_losses = []
    validation_losses = []
    spectral_losses = []
    validation_steps = []

    print(f"Starting training for {config.num_epochs} epochs...")

    for epoch in range(config.num_epochs):
        # Set to train mode
        pipeline.unet.train()
        epoch_loss = 0.0
        progress_bar = tqdm(
            total=len(train_dataloader),
            desc=f"Epoch {epoch+1}/{config.num_epochs}"
        )
        # Batch loop with error handling
        for step, batch in enumerate(train_dataloader):
            # Handle empty batches
            if batch is None or len(batch) == 0:
                progress_bar.update(1)
                continue
            try:
                # Process batch
                pixel_values = batch["image"].to(device, dtype=model_dtype)
                input_ids = batch["input_ids"].to(device)

                # Text embeddings
                with torch.no_grad():
                    text_embeddings = pipeline.text_encoder(input_ids)[0]
                    text_embeddings = text_embeddings.to(dtype=model_dtype)

                # Encode images
                with torch.no_grad():
                    # VAE encode
                    latents = pipeline.vae.encode(pixel_values).latent_dist.sample()
                    latents = latents * 0.18215

                # Sample noise
                noise = torch.randn_like(latents)
                batch_size = latents.shape[0]

                # Sample timesteps - use weighted timestep sampling for better audio quality
                # This samples more from early and late timesteps
                weights = torch.cat([
                    torch.linspace(1.0, 0.3, pipeline.scheduler.config.num_train_timesteps // 3),
                    torch.linspace(0.3, 0.1, pipeline.scheduler.config.num_train_timesteps // 3),
                    torch.linspace(0.1, 1.0, pipeline.scheduler.config.num_train_timesteps - 2*(pipeline.scheduler.config.num_train_timesteps // 3))
                ])
                prob = weights / weights.sum()
                timestep_indices = torch.multinomial(prob, batch_size, replacement=True)
                timesteps = timestep_indices.to(device).long()

                # Add noise
                noisy_latents = pipeline.scheduler.add_noise(latents, noise, timesteps)

                # Forward pass with mixed precision
                from torch.cuda.amp import autocast
                with autocast(enabled=use_mixed_precision):
                    # Predict noise
                    noise_pred = pipeline.unet(noisy_latents, timesteps, text_embeddings).sample
                    # Calculate regular loss
                    loss = torch.nn.functional.mse_loss(noise_pred, noise, reduction="mean")

                # Backpropagation
                if use_mixed_precision:
                    scaler.scale(loss).backward()
                else:
                    loss.backward()
                # Step optimizer after accumulation
                if (step + 1) % config.gradient_accumulation_steps == 0:
                    if use_mixed_precision:
                        scaler.unscale_(optimizer)

                    # Clip gradients for stability
                    torch.nn.utils.clip_grad_norm_(trainable_params, config.max_grad_norm)
                    if use_mixed_precision:
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        optimizer.step()

                    # Update LR with appropriate scheduler
                    lr_scheduler.step()

                    optimizer.zero_grad()
                    # Update tracking
                    global_step += 1
                    training_losses.append(loss.item())

                    # Save checkpoint periodically
                    if global_step % 500 == 0:
                        checkpoint_dir = os.path.join(output_dir, f"checkpoint-{global_step}")
                        os.makedirs(checkpoint_dir, exist_ok=True)
                        torch.save(pipeline.unet.state_dict(), os.path.join(checkpoint_dir, "unet.pt"))
                        print(f"\nSaved checkpoint at step {global_step}")

                    # Validation check with audio quality metrics
                    if global_step % config.val_check_interval == 0:
                        # Run validation
                        val_loss, spectral_loss = validate_audio_quality(
                            pipeline.unet, pipeline, val_dataloader, device, model_dtype, use_mixed_precision
                        )
                        validation_losses.append(val_loss)
                        spectral_losses.append(spectral_loss)
                        validation_steps.append(global_step)
                        print(f"\nValidation loss: {val_loss:.4f}, Spectral loss: {spectral_loss:.4f}")

                        # Save best model based on combined metric
                        combined_loss = val_loss + 0.1 * spectral_loss
                        if combined_loss < best_val_loss:
                            best_val_loss = combined_loss

                            # Save model
                            best_model_dir = os.path.join(output_dir, "best_model")
                            os.makedirs(best_model_dir, exist_ok=True)
                            torch.save(pipeline.unet.state_dict(), os.path.join(best_model_dir, "unet.pt"))
                            print(f"New best model saved (combined loss: {combined_loss:.4f})")

                # Update progress bar
                epoch_loss += loss.item()
                progress_bar.set_postfix({
                    "loss": f"{loss.item():.4f}",
                    "lr": f"{optimizer.param_groups[0]['lr']:.6f}"
                })
                progress_bar.update(1)
            except Exception as e:
                print(f"Training batch error: {e}")
                import traceback
                traceback.print_exc()
                progress_bar.update(1)
                continue

        # End of epoch
        progress_bar.close()
        avg_epoch_loss = epoch_loss / len(train_dataloader)
        print(f"Epoch {epoch+1}/{config.num_epochs} complete, avg loss: {avg_epoch_loss:.4f}")

        # Save checkpoint
        checkpoint_dir = os.path.join(output_dir, f"checkpoint-epoch-{epoch+1}")
        os.makedirs(checkpoint_dir, exist_ok=True)
        torch.save(pipeline.unet.state_dict(), os.path.join(checkpoint_dir, "unet.pt"))

    # Save final model
    final_dir = os.path.join(output_dir, "final_model")
    os.makedirs(final_dir, exist_ok=True)
    torch.save(pipeline.unet.state_dict(), os.path.join(final_dir, "unet.pt"))

    # Save training history
    history = {
        "training_loss": [float(x) for x in training_losses],
        "validation_loss": [float(x) for x in validation_losses],
        "spectral_loss": [float(x) for x in spectral_losses],
        "validation_steps": validation_steps,
        "best_val_loss": float(best_val_loss),
        "parameter_percentage": float(param_percentage)
    }

    with open(os.path.join(output_dir, "training_history.json"), "w") as f:
        json.dump(history, f, indent=2)

    # Plot training progress
    plt.figure(figsize=(12, 8))

    # Plot main losses
    plt.subplot(2, 1, 1)
    plt.plot(range(len(training_losses)), training_losses, alpha=0.5, label="Training Loss")

    if validation_losses:
        plt.plot(validation_steps, validation_losses, 'o-', color='red', label="Validation Loss")

    plt.title("Training Progress")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.legend()

    # Plot spectral loss
    if spectral_losses:
        plt.subplot(2, 1, 2)
        plt.plot(validation_steps, spectral_losses, 'o-', color='green', label="Spectral Loss")
        plt.title("Audio Quality Metric")
        plt.xlabel("Steps")
        plt.ylabel("Spectral Loss")
        plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "training_progress.png"))

    print(f"Training complete! Model saved to: {final_dir}")
    print(f"Parameter coverage: {param_percentage:.2%}")
    return final_dir

# Run the optimized training
model_path = optimized_training()

## 6. Ablation Study and Analysis

### Ablation Study and Analysis

Current execution uses: run_single_instrument_ablation_study()
Available alternative functions for different use cases:
- evaluate_model_ablation(): Simple baseline comparison
- evaluate_single_instrument_quality(): Focused single instrument testing
- evaluate_ablated_configuration(): Specific configuration testing
- run_application(): Launch Gradio interface for model use

### 6.1 Baseline Model Evaluation

In [13]:
def evaluate_model_ablation(model_path, test_prompts, output_dir=None, num_examples=5):
    """Simplified evaluation for ablation study - focused on key metrics"""
    if output_dir is None:
        output_dir = os.path.join(OUTPUT_PATH, f"ablation_eval_{time.strftime('%Y%m%d_%H%M%S')}")

    os.makedirs(output_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load models
    baseline_pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(device)

    test_pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(device)

    # Load test model weights
    test_pipeline.unet.load_state_dict(torch.load(model_path, map_location=device))

    # Simplified metrics tracking
    metrics = {
        "visual_diff": [],
        "spectral_contrast": [],
        "audio_quality": []
    }

    # Generate comparison samples
    for i, prompt in enumerate(test_prompts[:num_examples]):
        print(f"Testing prompt {i+1}/{num_examples}: '{prompt}'")

        # Generate with same seed
        seed = i + 42
        if device == "cuda":
            generator = torch.Generator(device="cuda").manual_seed(seed)
        else:
            generator = torch.Generator(device="cpu").manual_seed(seed)

        # Generate baseline
        baseline_img = baseline_pipeline(prompt, generator=generator).images[0]

        # Generate test
        if device == "cuda":
            generator = torch.Generator(device="cuda").manual_seed(seed)
        else:
            generator = torch.Generator(device="cpu").manual_seed(seed)
        test_img = test_pipeline(prompt, generator=generator).images[0]

        # Compute quick metrics
        baseline_np = np.array(baseline_img)
        test_np = np.array(test_img)

        # Visual difference
        visual_diff = np.mean(np.abs(baseline_np.astype(float) - test_np.astype(float)))
        metrics["visual_diff"].append(visual_diff)

        # Save comparison
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(baseline_img)
        axes[0].set_title("Baseline")
        axes[0].axis('off')

        axes[1].imshow(test_img)
        axes[1].set_title("Test Model")
        axes[1].axis('off')

        axes[2].imshow(np.abs(baseline_np.astype(float) - test_np.astype(float)))
        axes[2].set_title("Difference")
        axes[2].axis('off')

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"comparison_{i}.png"))
        plt.close()

    # Save summary metrics
    summary = {
        "mean_visual_diff": np.mean(metrics["visual_diff"]),
        "std_visual_diff": np.std(metrics["visual_diff"]),
    }

    with open(os.path.join(output_dir, "metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    return metrics, summary


### 6.2 Component-wise Ablation

In [14]:
def evaluate_single_instrument_quality(model_path, test_prompts, output_dir):
    """
    Evaluate model's ability to generate clean single instrument samples
    Focused on separation, clarity, and one-shot/loop quality
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Metrics specific to single instrument generation
    single_instrument_metrics = {
        "spectral_clarity": [],      # How focused is the frequency response
        "temporal_consistency": [],  # For loops: how consistent is the pattern
        "attack_quality": [],        # For one-shots: clean attack envelope
        "harmonic_purity": [],       # Single vs multiple instrument detection
        "background_noise": []       # Cleanliness of the sample
    }

    # Process each prompt
    for prompt in test_prompts:
        metrics = evaluate_single_source(prompt, model_path, device)
        for key, value in metrics.items():
            if key in single_instrument_metrics:
                single_instrument_metrics[key].append(value)

    return single_instrument_metrics

def evaluate_single_source(prompt, model_path=None, device=None, audio_data=None):
    """Evaluate how well a prompt generates a single instrument/vocal"""

    if audio_data is not None:
        # Skip the generation part and go straight to analysis
        pass
    else:
        # Generate audio
        spec_img, audio_bytes, duration, seed = generate_with_blend_ratio(
            prompt=prompt,
            model_path=model_path,
            blend_ratio=1.0,  # Full fine-tuned
            seed=42
        )

        # Convert to numpy for analysis
        audio_data = np.frombuffer(audio_bytes.getvalue(), dtype=np.int16).astype(float) / 32768.0

    # Move all metrics calculation OUTSIDE the if/else block
    metrics = {}

    # 1. Spectral clarity - measure frequency focus
    stft = np.abs(librosa.stft(audio_data, n_fft=2048))
    freq_variance = np.var(np.mean(stft, axis=1))
    metrics["spectral_clarity"] = 1.0 / (1.0 + freq_variance)

    # 2. Temporal consistency (for loops)
    if "loop" in prompt.lower():
        autocorr = librosa.autocorrelate(audio_data)
        peak_idx = np.argmax(autocorr[1:]) + 1
        loop_quality = autocorr[peak_idx] / autocorr[0]
        metrics["temporal_consistency"] = float(loop_quality)

    # 3. Attack quality (for one-shots)
    if "hit" in prompt.lower() or "shot" in prompt.lower():
        onset_env = librosa.onset.onset_strength(y=audio_data, sr=44100)
        onset_times = librosa.onset.onset_detect(onset_envelope=onset_env, sr=44100)
        if len(onset_times) > 0:
            attack_sharpness = onset_env[onset_times[0]] / np.mean(onset_env)
            metrics["attack_quality"] = min(1.0, attack_sharpness / 5.0)

    # 4. Harmonic purity - detect if multiple instruments
    harmonic_ratio = librosa.effects.harmonic(audio_data)
    percussive_ratio = librosa.effects.percussive(audio_data)
    purity = np.mean(np.abs(harmonic_ratio)) / (np.mean(np.abs(harmonic_ratio)) + np.mean(np.abs(percussive_ratio)))
    metrics["harmonic_purity"] = float(purity)

    # 5. Background noise level
    silence_threshold = np.percentile(np.abs(audio_data), 5)
    noise_ratio = np.mean(np.abs(audio_data[np.abs(audio_data) < silence_threshold]))
    metrics["background_noise"] = 1.0 - min(1.0, noise_ratio * 10)

    return metrics



def perform_component_ablation(base_model_path, output_dir=None):
    """
    Systematically disable LoRA components to understand their contribution
    """
    if output_dir is None:
        output_dir = os.path.join(OUTPUT_PATH, f"ablation_components_{time.strftime('%Y%m%d_%H%M%S')}")

    os.makedirs(output_dir, exist_ok=True)
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Component configurations to test
    component_configs = {
        "full_model": {
            "mid_block_attn": True,
            "up_block_attn": True,
            "down_block_attn": True,
            "cross_attn": True,
            "time_embed": True,
            "io_proj": True,
            "resnet": True
        },
        "no_mid_attn": {
            "mid_block_attn": False,
            "up_block_attn": True,
            "down_block_attn": True,
            "cross_attn": True,
            "time_embed": True,
            "io_proj": True,
            "resnet": True
        },
        "no_cross_attn": {
            "mid_block_attn": True,
            "up_block_attn": True,
            "down_block_attn": True,
            "cross_attn": False,
            "time_embed": True,
            "io_proj": True,
            "resnet": True
        },
        "no_time_embed": {
            "mid_block_attn": True,
            "up_block_attn": True,
            "down_block_attn": True,
            "cross_attn": True,
            "time_embed": False,
            "io_proj": True,
            "resnet": True
        },
        "attn_only": {
            "mid_block_attn": True,
            "up_block_attn": True,
            "down_block_attn": True,
            "cross_attn": True,
            "time_embed": False,
            "io_proj": False,
            "resnet": False
        }
    }

    # Test prompts for single instrument generation
    test_prompts = [
        "single drum hit, clean kick, one shot",
        "isolated piano key, C note",
        "clean guitar string, single pluck",
        "synthesizer pad, single chord Cm",
        "vocal hit, single word 'ah'"
    ]

    results = {}

    for config_name, config in component_configs.items():
        print(f"\nTesting configuration: {config_name}")

        # Create ablated model
        pipeline, ablator = create_ablated_model(base_model_path, config, device)

        # Evaluate this configuration
        config_output_dir = os.path.join(output_dir, config_name)
        metrics = evaluate_ablated_pipeline(
            pipeline,
            test_prompts,
            config_output_dir,
            config_name
        )

        # Clean up hooks
        ablator.remove_hooks()

        results[config_name] = metrics

    # Create component importance visualization
    visualize_component_importance(results, output_dir)

    return results

def create_ablated_model(base_model_path, config, device):
    """Create a model with specific components disabled through functional ablation"""
    # Load models
    pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(device)

    # Load fine-tuned weights
    unet_path = os.path.join(base_model_path, "unet.pt")
    pipeline.unet.load_state_dict(torch.load(unet_path, map_location=device))

    # Create a custom forward hook for ablation
    class ComponentAblator:
        def __init__(self, config):
            self.config = config
            self.hooks = []

        def register_ablation_hooks(self, model):
            """Register hooks to ablate specific components during forward pass"""

            # Mid-block attention ablation
            if not self.config["mid_block_attn"]:
                for attn in model.mid_block.attentions:
                    hook = attn.register_forward_hook(self.zero_output_hook)
                    self.hooks.append(hook)

            # Up block attention ablation
            if not self.config["up_block_attn"]:
                for block in model.up_blocks:
                    if hasattr(block, 'attentions'):
                        for attn in block.attentions:
                            hook = attn.register_forward_hook(self.zero_output_hook)
                            self.hooks.append(hook)

            # Down block attention ablation
            if not self.config["down_block_attn"]:
                for block in model.down_blocks:
                    if hasattr(block, 'attentions'):
                        for attn in block.attentions:
                            hook = attn.register_forward_hook(self.zero_output_hook)
                            self.hooks.append(hook)

            # Cross-attention ablation (attn2)
            if not self.config["cross_attn"]:
                for block in model.up_blocks + model.down_blocks:
                    if hasattr(block, 'attentions'):
                        for attn in block.attentions:
                            if hasattr(attn, 'transformer_blocks'):
                                for transformer in attn.transformer_blocks:
                                    if hasattr(transformer, 'attn2'):
                                        hook = transformer.attn2.register_forward_hook(self.zero_output_hook)
                                        self.hooks.append(hook)

            # Time embedding ablation
            if not self.config["time_embed"]:
                if hasattr(model, 'time_embedding'):
                    hook = model.time_embedding.register_forward_hook(self.zero_output_hook)
                    self.hooks.append(hook)

            # ResNet ablation
            if not self.config["resnet"]:
                for block in model.up_blocks + model.down_blocks:
                    if hasattr(block, 'resnets'):
                        for resnet in block.resnets:
                            hook = resnet.register_forward_hook(self.zero_output_hook)
                            self.hooks.append(hook)

                # Mid-block resnets
                if hasattr(model.mid_block, 'resnets'):
                    for resnet in model.mid_block.resnets:
                        hook = resnet.register_forward_hook(self.zero_output_hook)
                        self.hooks.append(hook)

            # I/O projection ablation
            if not self.config["io_proj"]:
                if hasattr(model, 'conv_in'):
                    hook = model.conv_in.register_forward_hook(self.zero_output_hook)
                    self.hooks.append(hook)
                if hasattr(model, 'conv_out'):
                    hook = model.conv_out.register_forward_hook(self.zero_output_hook)
                    self.hooks.append(hook)

        def zero_output_hook(self, module, input, output):
            """Hook that zeros the output of a module"""
            if isinstance(output, torch.Tensor):
                return torch.zeros_like(output)
            elif isinstance(output, tuple):
                return tuple(torch.zeros_like(o) if isinstance(o, torch.Tensor) else o for o in output)
            return output

        def remove_hooks(self):
            """Clean up hooks after use"""
            for hook in self.hooks:
                hook.remove()

    # Apply ablation hooks
    ablator = ComponentAblator(config)
    ablator.register_ablation_hooks(pipeline.unet)

    return pipeline, ablator


def evaluate_ablated_pipeline(pipeline, test_prompts, output_dir, config_name):
    """Evaluate an ablated pipeline directly"""
    os.makedirs(output_dir, exist_ok=True)

    metrics = {
        "spectrogram_similarity": [],
        "audio_characteristics": [],
        "prompt_adherence": [],
        "visual_diff": []
    }

    # Load baseline pipeline for comparison
    baseline_pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(pipeline.device)

    for i, prompt in enumerate(test_prompts):
        try:
            # Generate with ablated pipeline
            ablated_img = pipeline(prompt, num_inference_steps=25).images[0]

            # Generate with baseline
            baseline_img = baseline_pipeline(prompt, num_inference_steps=25).images[0]

            # Calculate metrics
            baseline_np = np.array(baseline_img).astype(float)
            ablated_np = np.array(ablated_img).astype(float)

            # Visual difference
            mse = np.mean((baseline_np - ablated_np) ** 2)
            metrics["visual_diff"].append(mse)
            metrics["spectrogram_similarity"].append(1.0 / (1.0 + mse))

            # Save comparison
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
            ax1.imshow(baseline_img)
            ax1.set_title("Baseline")
            ax1.axis('off')
            ax2.imshow(ablated_img)
            ax2.set_title(f"{config_name}")
            ax2.axis('off')
            plt.savefig(os.path.join(output_dir, f"comparison_{i}.png"))
            plt.close()

            # Additional audio metrics can be added here
            metrics["audio_characteristics"].append(0.5)  # Placeholder
            metrics["prompt_adherence"].append(0.85)  # Placeholder

        except Exception as e:
            print(f"Error evaluating {config_name} with prompt '{prompt}': {e}")
            continue

    # Calculate summary metrics
    summary = {}
    for key, values in metrics.items():
        if values:
            summary[f"mean_{key}"] = float(np.mean(values))
            summary[f"std_{key}"] = float(np.std(values))

    return summary



def evaluate_ablated_configuration(pipeline, ablator, test_prompts, config_name):
    """Evaluate an ablated model configuration on single instrument generation"""
    metrics = {
        "single_instrument_quality": [],
        "prompt_specificity": [],
        "temporal_characteristics": []
    }

    for prompt in test_prompts:
        # Generate with ablated model
        try:
            result = pipeline(prompt, num_inference_steps=25).images[0]

            # Convert to audio and evaluate
            audio_bytes, duration = wav_bytes_from_spectrogram_image(result)
            audio_data = np.frombuffer(audio_bytes.getvalue(), dtype=np.int16).astype(float) / 32768.0

            # Evaluate single instrument metrics
            single_metrics = evaluate_single_source(prompt, audio_data=audio_data)

            # Store results
            for key, value in single_metrics.items():
                if key in metrics:
                    metrics[key].append(value)

        except Exception as e:
            print(f"Error evaluating {config_name} with prompt '{prompt}': {e}")
            continue

    # Clean up ablation hooks
    ablator.remove_hooks()

    # Calculate averages
    avg_metrics = {}
    for key, values in metrics.items():
        if values:
            avg_metrics[f"avg_{key}"] = np.mean(values)
            avg_metrics[f"std_{key}"] = np.std(values)

    return avg_metrics


def visualize_component_importance(results, output_dir):
    """Create visualization showing importance of each component"""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))

    # Component rankings
    configs = list(results.keys())
    scores = [results[config]["mean_visual_diff"] for config in configs]

    # Sort by impact (higher diff = more important)
    sorted_pairs = sorted(zip(configs, scores), key=lambda x: x[1], reverse=True)
    sorted_configs, sorted_scores = zip(*sorted_pairs)

    # Bar chart
    bars = ax1.bar(range(len(sorted_configs)), sorted_scores, color='skyblue')
    ax1.set_xticks(range(len(sorted_configs)))
    ax1.set_xticklabels(sorted_configs, rotation=45, ha='right')
    ax1.set_ylabel('Average Visual Difference')
    ax1.set_title('Component Importance (Higher = More Important)')

    # Add value labels
    for bar, score in zip(bars, sorted_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.2f}', ha='center', va='bottom')

    # Relative importance chart
    full_model_score = results["full_model"]["mean_visual_diff"]
    relative_impacts = {}

    for config in configs:
        if config != "full_model":
            impact = (full_model_score - results[config]["mean_visual_diff"]) / full_model_score
            relative_impacts[config] = impact

    configs_rel = list(relative_impacts.keys())
    impacts = list(relative_impacts.values())

    ax2.bar(range(len(configs_rel)), impacts, color='lightcoral')
    ax2.set_xticks(range(len(configs_rel)))
    ax2.set_xticklabels(configs_rel, rotation=45, ha='right')
    ax2.set_ylabel('Relative Impact')
    ax2.set_title('Component Impact Relative to Full Model')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'component_importance.png'), dpi=300)
    plt.show()
    plt.close()

### 6.3 Blended Model Analysis (Latent Space + Pixel Level)

In [15]:
def generate_with_blend_ratio(prompt, model_path, blend_ratio=0.5, blend_type="latent",
                            guidance_scale=7.5, steps=25, seed=None):
    """
    Generate audio with blended models using either latent or pixel blending

    Args:
        prompt: Text prompt
        model_path: Path to fine-tuned model
        blend_ratio: 1.0 = fully fine-tuned, 0.0 = fully baseline
        blend_type: "latent" or "pixel"
        guidance_scale: Guidance strength
        steps: Inference steps
        seed: Random seed for reproducibility
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load models
    base_pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(device)

    ft_pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(device)

    # Load fine-tuned weights
    if os.path.isdir(model_path):
      unet_path = os.path.join(model_path, "unet.pt")
    else:
      unet_path = model_path

    ft_pipeline.unet.load_state_dict(torch.load(unet_path, map_location=device))

    # Set seed
    if seed is None:
        seed = random.randint(0, 2**32-1)

    if device == "cuda":
        generator = torch.Generator(device="cuda").manual_seed(seed)
    else:
        generator = torch.Generator(device="cpu").manual_seed(seed)

    if blend_type == "latent":
        # Latent space blending
        blended_img = blend_in_latent_space(
            base_pipeline, ft_pipeline, prompt, blend_ratio,
            guidance_scale, steps, generator, device
        )
    else:
        # Pixel space blending (your original method)
        # Generate with base model
        if device == "cuda":
            base_gen = torch.Generator(device="cuda").manual_seed(seed)
        else:
            base_gen = torch.Generator(device="cpu").manual_seed(seed)
        base_img = base_pipeline(
            prompt=prompt,
            guidance_scale=guidance_scale,
            num_inference_steps=steps,
            generator=base_gen
        ).images[0]

        # Generate with fine-tuned model
        if device == "cuda":
            ft_gen = torch.Generator(device="cuda").manual_seed(seed)
        else:
            ft_gen = torch.Generator(device="cpu").manual_seed(seed)
        ft_img = ft_pipeline(
            prompt=prompt,
            guidance_scale=guidance_scale,
            num_inference_steps=steps,
            generator=ft_gen
        ).images[0]

        # Blend images (pixel-level)
        base_array = np.array(base_img).astype(float)
        ft_array = np.array(ft_img).astype(float)
        blended_array = (blend_ratio * base_array + (1-blend_ratio) * ft_array).astype(np.uint8)
        blended_img = Image.fromarray(blended_array)

    # Convert to audio
    audio_bytes, duration = wav_bytes_from_spectrogram_image(blended_img)

    return blended_img, audio_bytes, duration, seed

def blend_in_latent_space(base_pipeline, ft_pipeline, prompt, blend_ratio,
                         guidance_scale, steps, generator, device):
    """Blend models in latent space for smoother transitions"""

    # Get text embeddings
    text_embeddings = get_text_embeddings(base_pipeline, prompt, device)

    # Initialize latents
    latents = torch.randn(
        (1, base_pipeline.unet.in_channels, 64, 64),
        generator=generator,
        device=device
    )

    # Set scheduler
    base_pipeline.scheduler.set_timesteps(steps, device=device)
    timesteps = base_pipeline.scheduler.timesteps

    # Denoising loop with blended predictions
    for i, t in enumerate(timesteps):
        # Expand latents for guidance
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = base_pipeline.scheduler.scale_model_input(latent_model_input, t)

        # Predict noise with base model
        with torch.no_grad():
            noise_pred_base = base_pipeline.unet(
                latent_model_input, t, encoder_hidden_states=text_embeddings
            ).sample

        # Predict noise with fine-tuned model
        with torch.no_grad():
            noise_pred_ft = ft_pipeline.unet(
                latent_model_input, t, encoder_hidden_states=text_embeddings
            ).sample

        # Blend predictions
        noise_pred = noise_pred_base * (1 - blend_ratio) + noise_pred_ft * blend_ratio

        # Perform classifier-free guidance
        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

        # Compute previous noisy sample
        latents = base_pipeline.scheduler.step(noise_pred, t, latents).prev_sample

    # Decode latents
    latents = 1 / 0.18215 * latents
    with torch.no_grad():
        image = base_pipeline.vae.decode(latents).sample

    # Convert to PIL
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 1).numpy()
    image = (image * 255).astype(np.uint8)
    blended_img = Image.fromarray(image[0])

    return blended_img

def get_text_embeddings(pipeline, prompt, device):
    """Get text embeddings for a prompt"""
    text_inputs = pipeline.tokenizer(
        prompt,
        padding="max_length",
        max_length=pipeline.tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )

    with torch.no_grad():
        text_embeddings = pipeline.text_encoder(text_inputs.input_ids.to(device))[0]

    # Duplicate for classifier-free guidance
    uncond_input = pipeline.tokenizer(
        [""],
        padding="max_length",
        max_length=pipeline.tokenizer.model_max_length,
        return_tensors="pt",
    )

    with torch.no_grad():
        uncond_embeddings = pipeline.text_encoder(uncond_input.input_ids.to(device))[0]

    # Concatenate for classifier-free guidance
    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

    return text_embeddings

def find_optimal_model_configuration(component_results, blend_results, criteria='single_instrument'):
    """
    Determine the optimal model configuration based on specific criteria
    Priority: Single instrument/vocal generation quality
    """
    print("\n=== Model Configuration Analysis ===")

    # Analyze component ablation results
    component_scores = {}
    for config_name, results in component_results.items():
        if config_name != "full_model":
            # Calculate composite score for single instrument generation
            score = 0
            weights = {
                'spectral_clarity': 0.3,
                'harmonic_purity': 0.3,
                'attack_quality': 0.2,
                'background_noise': 0.2
            }

            for metric, weight in weights.items():
                if f"avg_{metric}" in results:
                    score += results[f"avg_{metric}"] * weight

            component_scores[config_name] = score

    # Find best component configuration
    best_component_config = max(component_scores.items(), key=lambda x: x[1])
    print(f"\nBest Component Configuration: {best_component_config[0]}")
    print(f"Score: {best_component_config[1]:.4f}")

    # Analyze blend ratios
    blend_scores = {}
    for ratio, results in blend_results.items():
        if 'audio_quality' in results:
            # Adjust for single instrument preference
            single_instrument_bonus = 0
            if ratio >= 0.7:  # Higher fine-tuned ratio generally better for specificity
                single_instrument_bonus = 0.1 * ratio

            blend_scores[ratio] = results['audio_quality'] + single_instrument_bonus

    # Find optimal blend ratio
    if blend_scores:
        optimal_blend = max(blend_scores.items(), key=lambda x: x[1])
        print(f"\nOptimal Blend Ratio: {optimal_blend[0]:.2f}")
        print(f"Score: {optimal_blend[1]:.4f}")

    # Create recommendation
    recommendation = {
        "optimal_component_config": best_component_config[0],
        "optimal_blend_ratio": optimal_blend[0] if blend_scores else 0.75,
        "component_importance_ranking": sorted(component_scores.items(), key=lambda x: x[1], reverse=True),
        "recommended_for_gui": {
            "config": "minimal_with_cross_attn",  # For efficiency
            "blend_ratio": optimal_blend[0] if blend_scores else 0.75,
            "inference_steps": 25,  # Balance speed vs quality
            "guidance_scale": 8.0   # Higher for better prompt adherence
        }
    }

    # Save recommendation
    output_dir = os.path.join(OUTPUT_PATH, "optimal_config")
    os.makedirs(output_dir, exist_ok=True)

    with open(os.path.join(output_dir, "optimal_configuration.json"), "w") as f:
        json.dump(recommendation, f, indent=2)

    return recommendation


### 6.4 Quantitative Results

In [16]:
def generate_quantitative_analysis(ablation_results, blend_results, output_dir):
    """Generate analysis focused on single instrument generation quality"""

    analysis_dir = os.path.join(output_dir, "single_instrument_analysis")
    os.makedirs(analysis_dir, exist_ok=True)

    # Create comprehensive visualization
    fig = plt.figure(figsize=(20, 15))
    gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

    # 1. Single Instrument Quality Comparison
    ax1 = fig.add_subplot(gs[0, :])

    configs = list(ablation_results.keys())
    single_instrument_scores = []

    for config in configs:
        if config in ablation_results:
            # Calculate composite score for single instrument quality
            score = 0
            metrics = ['spectral_clarity', 'harmonic_purity', 'attack_quality', 'background_noise']
            valid_metrics = 0

            for metric in metrics:
                if f"avg_{metric}" in ablation_results[config]:
                    score += ablation_results[config][f"avg_{metric}"]
                    valid_metrics += 1

            if valid_metrics > 0:
                single_instrument_scores.append(score / valid_metrics)
            else:
                single_instrument_scores.append(0)

    bars = ax1.bar(range(len(configs)), single_instrument_scores, color=['steelblue' if c != 'full_model' else 'gold' for c in configs])
    ax1.set_xticks(range(len(configs)))
    ax1.set_xticklabels(configs, rotation=45, ha='right')
    ax1.set_ylabel('Single Instrument Quality Score')
    ax1.set_title('Component Configurations: Single Instrument Generation Quality')

    # Add value labels
    for bar, score in zip(bars, single_instrument_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom')

    # 2. Specific Metric Breakdown
    ax2 = fig.add_subplot(gs[1, :])

    metrics_breakdown = {}
    for metric in ['spectral_clarity', 'harmonic_purity', 'attack_quality', 'background_noise']:
        metric_values = []
        for config in configs:
            if f"avg_{metric}" in ablation_results.get(config, {}):
                metric_values.append(ablation_results[config][f"avg_{metric}"])
            else:
                metric_values.append(0)
        metrics_breakdown[metric] = metric_values

    x = np.arange(len(configs))
    width = 0.2
    multiplier = 0

    for metric, values in metrics_breakdown.items():
        offset = width * multiplier
        bars = ax2.bar(x + offset, values, width, label=metric.replace('_', ' ').title())
        multiplier += 1

    ax2.set_xlabel('Configuration')
    ax2.set_ylabel('Metric Score')
    ax2.set_title('Detailed Single Instrument Metrics by Configuration')
    ax2.set_xticks(x + width * 1.5)
    ax2.set_xticklabels(configs, rotation=45, ha='right')
    ax2.legend(loc='upper left', bbox_to_anchor=(0, 1))
    ax2.grid(True, alpha=0.3)

    # 3. Optimal Configuration Recommendation
    ax3 = fig.add_subplot(gs[2, :])
    ax3.axis('off')

    # Find optimal configuration
    optimal_config = max(zip(configs, single_instrument_scores), key=lambda x: x[1])

    recommendation_text = f"""Optimal Model Configuration for Single Instrument Generation:

Configuration: {optimal_config[0]}
Quality Score: {optimal_config[1]:.4f}

Key Findings:
- Best spectral clarity: {max(metrics_breakdown['spectral_clarity']):.3f}
- Best harmonic purity: {max(metrics_breakdown['harmonic_purity']):.3f}
- Best attack quality: {max(metrics_breakdown['attack_quality']):.3f}
- Lowest background noise: {max(metrics_breakdown['background_noise']):.3f}

Recommendations for GUI Integration:
- Use {optimal_config[0]} configuration
- Blend ratio: {max(blend_results.items(), key=lambda x: x[1].get('audio_quality', 0))[0]:.2f}
- Focus on instrument-specific prompts
- Use higher guidance scales (8.0-10.0) for better separation
- Shorter inference steps (25-30) for single instruments"""

    ax3.text(0.05, 0.95, recommendation_text, transform=ax3.transAxes, fontsize=12,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

    # 4. Component Contribution Analysis
    ax4 = fig.add_subplot(gs[3, :])

    # Calculate importance based on quality drop when ablated
    full_model_score = single_instrument_scores[configs.index('full_model')] if 'full_model' in configs else 0
    component_importance = {}

    for config, score in zip(configs, single_instrument_scores):
        if config != 'full_model':
            importance = full_model_score - score
            component_importance[config] = max(0, importance)  # Ensure non-negative

    sorted_importance = sorted(component_importance.items(), key=lambda x: x[1], reverse=True)

    if sorted_importance:
        components, importances = zip(*sorted_importance)
        colors = plt.cm.viridis(np.linspace(0, 1, len(components)))
        bars = ax4.bar(range(len(components)), importances, color=colors)
        ax4.set_xticks(range(len(components)))
        ax4.set_xticklabels(components, rotation=45, ha='right')
        ax4.set_ylabel('Contribution to Single Instrument Quality')
        ax4.set_title('Component Importance for Single Instrument Generation')

        for bar, imp in zip(bars, importances):
            ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{imp:.3f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.savefig(os.path.join(analysis_dir, 'single_instrument_analysis.png'), dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    return optimal_config, recommendation_text



### 6.5 Audio Gallery (Interactive with Saved Output)

Please view the [Audio Gallery HTML file here](https://adinsegall.github.io/riffusion-lora/) which contains the embedded auido output examples

In [17]:
import shutil
def create_audio_gallery(model_path, output_dir=None, save_audio=True):
    """Create interactive audio gallery with saved outputs for persistent playback"""

    if output_dir is None:
        output_dir = os.path.join(OUTPUT_PATH, f"audio_gallery_{time.strftime('%Y%m%d_%H%M%S')}")

    os.makedirs(output_dir, exist_ok=True)

    # Define gallery structure based on the dataset to showoff model performance a bit better
    gallery_structure = {
        "Drum Loops": [
            "drum loop, techno style, 128 BPM, punchy kick and hi-hats, electronic",
            "drum sound, trap style, 140 BPM, heavy 808 and snares, hip hop",
            "percussion loop, dubstep style, 150 BPM, aggressive drums, electronic",
            "drum pattern, house music, 126 BPM, four-on-floor kick, professional"
        ],
        "Melody Loops": [
            "melodic sound, jazz style, 95 BPM, in key Eb minor, smooth chord progression",
            "synth lead, future bass style, 150 BPM, bright harmonics, modulated",
            "piano melody, lofi hip hop style, 85 BPM, in key A minor, vintage sound",
            "string ensemble, cinematic style, 90 BPM, emotional dynamics, layered"
        ],
        "Bass Loops": [
            "bass sound, bass house style, 128 BPM, deep sub bass, four-on-floor",
            "bass loop, dubstep style, 150 BPM, heavy wobble bass, riddim",
            "melodic bass, deep house style, 125 BPM, rolling bassline, warm",
            "synth bass, techno style, 130 BPM, punchy and clean, electronic"
        ],
        "One-Shots": [
            "drum sound, techno style, single kick hit, punchy and clean",
            "percussion hit, trap style, snare shot, bright and snappy",
            "bass hit, dubstep style, sub drop impact, massive presence",
            "synth stab, house music style, chord hit, punchy attack"
        ],
        "FX & Textures": [
            "ambient pad, cinematic style, ethereal atmosphere, evolving texture",
            "sound effect, electronic style, riser sweep, build up tension",
            "background texture, lofi style, vinyl atmosphere, subtle noise",
            "transition effect, dubstep style, whoosh sweep, downlifter"
        ]
    }

    # Generate all audio samples
    generated_samples = {}
    audio_files = {}

    print("Generating audio gallery samples...")

    for category, prompts in gallery_structure.items():
        print(f"\nGenerating {category}...")
        generated_samples[category] = []
        audio_files[category] = []

        for i, prompt in enumerate(prompts):
            # Generate with different blend ratios
            for blend_ratio in [0.0, 0.5, 1.0]:
                sample_id = f"{category}_{i}_{blend_ratio}"

                # Generate audio
                spec_img, audio_bytes, duration, seed = generate_with_blend_ratio(
                    prompt=prompt,
                    model_path=model_path,
                    blend_ratio=blend_ratio,
                    blend_type="latent",
                    seed=42 + i  # Consistent seed for comparison
                )

                # Save audio file if requested
                if save_audio:
                    audio_filename = f"{sample_id.replace(' ', '_')}.wav"
                    audio_path = os.path.join(output_dir, audio_filename)

                    with open(audio_path, 'wb') as f:
                        f.write(audio_bytes.getbuffer())

                    audio_files[category].append(audio_path)

                # Store metadata
                sample_info = {
                    "prompt": prompt,
                    "blend_ratio": blend_ratio,
                    "seed": seed,
                    "audio_path": audio_path if save_audio else None,
                    "spectrogram": spec_img
                }

                generated_samples[category].append(sample_info)

    # Create interactive gallery HTML
    html_gallery = create_gallery_html(generated_samples, output_dir)

    # Create Jupyter widget interface
    gallery_widget = create_gallery_widget(generated_samples, audio_files)

    # Save gallery metadata
    gallery_metadata = {
        "categories": list(gallery_structure.keys()),
        "total_samples": sum(len(samples) for samples in generated_samples.values()),
        "output_directory": output_dir,
        "creation_time": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    with open(os.path.join(output_dir, "gallery_metadata.json"), "w") as f:
        json.dump(gallery_metadata, f, indent=2)

    return gallery_widget, generated_samples, html_gallery

def create_gallery_widget(generated_samples, audio_files):
    """Create interactive widget for audio gallery"""

    # Create dropdowns
    category_options = list(generated_samples.keys())
    category_dropdown = widgets.Dropdown(
        options=category_options,
        description='Category:'
    )

    # Create sample selector
    sample_dropdown = widgets.Dropdown(
        description='Sample:'
    )

    # Create blend selector
    blend_dropdown = widgets.Dropdown(
        options=[('Baseline (0%)', 0.0), ('Blended (50%)', 0.5), ('Fine-tuned (100%)', 1.0)],
        description='Model:'
    )

    # Create output areas
    spectrogram_output = widgets.Output()
    audio_output = widgets.Output()
    info_output = widgets.Output()

    def update_sample_options(change):
        category = change['new']
        samples = generated_samples[category]

        # Get unique prompts
        unique_prompts = []
        for sample in samples:
            if sample['blend_ratio'] == 0.0:  # Only show baseline samples
                unique_prompts.append(sample['prompt'])

        sample_dropdown.options = unique_prompts
        if unique_prompts:
            sample_dropdown.value = unique_prompts[0]

    def update_display(change=None):
        with spectrogram_output:
            spectrogram_output.clear_output()

        with audio_output:
            audio_output.clear_output()

        with info_output:
            info_output.clear_output()

        category = category_dropdown.value
        prompt = sample_dropdown.value
        blend_ratio = blend_dropdown.value

        # Find matching sample
        for sample in generated_samples[category]:
            if sample['prompt'] == prompt and sample['blend_ratio'] == blend_ratio:
                # Display spectrogram
                with spectrogram_output:
                    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
                    ax.imshow(sample['spectrogram'])
                    ax.axis('off')
                    ax.set_title(f"Spectrogram - {blend_dropdown.label}")
                    plt.tight_layout()
                    plt.show()
                    plt.close()

                # Display audio player
                with audio_output:
                    if sample['audio_path'] and os.path.exists(sample['audio_path']):
                        display(Audio(sample['audio_path']))
                    else:
                        print("Audio file not found")

                # Display info
                with info_output:
                    print(f"Prompt: {prompt}")
                    print(f"Model: {blend_dropdown.label}")
                    print(f"Seed: {sample['seed']}")
                    if sample['audio_path']:
                        print(f"File: {os.path.basename(sample['audio_path'])}")

                break

    # Connect dropdown callbacks
    category_dropdown.observe(update_sample_options, names='value')
    sample_dropdown.observe(update_display, names='value')
    blend_dropdown.observe(update_display, names='value')

    # Initialize
    if category_options:
        category_dropdown.value = category_options[0]
        update_sample_options({'new': category_options[0]})
        update_display()

    # Layout
    gallery_widget = widgets.VBox([
        widgets.HTML("<h3>Audio Gallery</h3>"),
        widgets.HBox([category_dropdown, sample_dropdown, blend_dropdown]),
        info_output,
        spectrogram_output,
        audio_output
    ])

    return gallery_widget

def create_gallery_html(generated_samples, output_dir):
    """Create standalone HTML gallery with embedded audio"""
    import base64

    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <title>Riffusion Ablation Gallery</title>
        <style>
            body { font-family: Arial, sans-serif; max-width: 1200px; margin: 0 auto; padding: 20px; }
            .category { margin-bottom: 40px; }
            .sample { margin-bottom: 20px; border: 1px solid #ddd; padding: 10px; }
            .audio-row { display: flex; gap: 20px; align-items: center; flex-wrap: wrap; }
            .blend-type { font-weight: bold; margin-bottom: 5px; }
            audio { width: 300px; margin: 5px 0; }
            h2 { color: #333; border-bottom: 2px solid #333; padding-bottom: 5px; }
            h3 { color: #666; margin-bottom: 10px; }
            .audio-container { margin: 5px; }
            .download-link {
                font-size: 0.9em;
                color: #666;
                text-decoration: none;
                padding: 5px 10px;
                background: #f0f0f0;
                border-radius: 3px;
                margin-top: 5px;
                display: inline-block;
            }
            .download-link:hover { background: #e0e0e0; }
        </style>
    </head>
    <body>
        <h1>Riffusion Fine-tuning Ablation Gallery</h1>
        <p>Generated on: """ + time.strftime("%Y-%m-%d %H:%M:%S") + """</p>
        <p class="note">Audio files are embedded directly in this HTML - you can save this file and share it!</p>
    """

    for category, samples in generated_samples.items():
        html_content += f"<div class='category'><h2>{category}</h2>"

        # Group samples by prompt
        prompts = {}
        for sample in samples:
            prompt = sample['prompt']
            if prompt not in prompts:
                prompts[prompt] = []
            prompts[prompt].append(sample)

        for prompt, prompt_samples in prompts.items():
            html_content += f"<div class='sample'><h3>{prompt}</h3><div class='audio-row'>"

            for sample in prompt_samples:
                if sample['audio_path'] and os.path.exists(sample['audio_path']):
                    # Read audio file and encode as base64
                    with open(sample['audio_path'], 'rb') as audio_file:
                        audio_data = audio_file.read()
                        audio_base64 = base64.b64encode(audio_data).decode('utf-8')

                    blend_label = f"Blend: {sample['blend_ratio']:.0%}"

                    html_content += f"""
                    <div class='audio-container'>
                        <p class='blend-type'>{blend_label}</p>
                        <audio controls>
                            <source src="data:audio/wav;base64,{audio_base64}" type="audio/wav">
                            Your browser does not support the audio element.
                        </audio>
                        <br><a href="data:audio/wav;base64,{audio_base64}" download="riffusion_{prompt[:20].replace(' ', '_')}_blend{sample['blend_ratio']:.0%}.wav" class="download-link">Download WAV</a>
                    </div>
                    """

            html_content += "</div></div>"

        html_content += "</div>"

    html_content += """
        <script>
            // Ensure all audio elements are properly loaded
            document.querySelectorAll('audio').forEach(audio => {
                audio.addEventListener('loadedmetadata', function() {
                    console.log('Audio loaded successfully');
                });
                audio.addEventListener('error', function(e) {
                    console.error('Audio error:', e);
                    this.parentElement.innerHTML += '<p style="color: red; font-size: 0.8em;">Error loading audio</p>';
                });
            });

            // Add download functionality
            document.querySelectorAll('.download-link').forEach(link => {
                link.addEventListener('click', function(e) {
                    // Small delay to ensure download works
                    setTimeout(() => {
                        console.log('Download initiated');
                    }, 100);
                });
            });
        </script>
    </body>
    </html>
    """

    # Save HTML file
    html_path = os.path.join(output_dir, "gallery_standalone.html")
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"Standalone gallery created: {html_path}")
    print("This HTML file can be shared directly - all audio is embedded!")
    return html_path


def generate_final_report(component_results, blend_results, output_dir, optimal_config):
    """Generate comprehensive final report with optimal configuration recommendations"""

    report_path = os.path.join(output_dir, "ablation_study_report.md")

    with open(report_path, 'w') as f:
        f.write("# Riffusion Fine-tuning Ablation Study Report\n\n")
        f.write(f"Generated on: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        f.write("## Executive Summary\n\n")
        f.write("This comprehensive ablation study analyzes the impact of different model components ")
        f.write("and blending strategies on Riffusion's ability to generate professional music production samples.\n\n")

        f.write("## Key Findings\n\n")

        # Component importance
        f.write("### Component Importance Ranking\n\n")
        full_score = component_results["full_model"]["mean_visual_diff"]
        importance_scores = []

        for config, results in component_results.items():
            if config != "full_model":
                importance = (full_score - results["mean_visual_diff"]) / full_score
                importance_scores.append((config, importance))

        importance_scores.sort(key=lambda x: x[1], reverse=True)

        for i, (component, score) in enumerate(importance_scores, 1):
            f.write(f"{i}. **{component.replace('_', ' ').title()}**: {score:.3f}\n")

        # Add optimal configuration section using the optimal_config parameter
        f.write("\n### Optimal Configuration\n\n")
        f.write(f"- **Parameter Coverage**: ~8% of UNet parameters\n")
        f.write(f"- **Key Components**: {', '.join([comp for comp, _ in importance_scores[:3]])}\n")
        f.write(f"- **Optimal Component Config**: {optimal_config['optimal_component_config']}\n")
        f.write(f"- **Optimal Blend Ratio**: {optimal_config['optimal_blend_ratio']:.2f}\n")

        # Blending analysis
        f.write("\n### Blending Strategy Results\n\n")
        best_ratio = max(blend_results.items(), key=lambda x: x[1]['audio_quality'])[0]
        f.write(f"- **Optimal Blend Ratio**: {best_ratio:.0%} fine-tuned, {(1-best_ratio):.0%} baseline\n")
        f.write(f"- **Recommended Approach**: Latent space blending for smoother transitions\n\n")

        f.write("## Technical Details\n\n")
        f.write("### Training Configuration\n\n")
        f.write("```\n")
        f.write("- Learning Rate: 5e-5\n")
        f.write("- Batch Size: 4\n")
        f.write("- Epochs: 30\n")
        f.write("- LoRA Rank: 64\n")
        f.write("- LoRA Alpha: 128\n")
        f.write("```\n\n")

        f.write("### Dataset Characteristics\n\n")
        f.write("The model was fine-tuned on the WAProd Sample Pack, featuring:\n")
        f.write("- Professional EDM, Dubstep, Trap, and Techno samples\n")
        f.write("- Properly labeled loops and one-shots\n")
        f.write("- Genre-specific sound design elements\n")
        f.write("- Consistent BPM and key labeling\n\n")

        f.write("## Recommendations\n\n")
        f.write("1. **For production use**: Use 75% fine-tuned blend for best quality\n")
        f.write("2. **For experimentation**: Start with component ablation results\n")
        f.write(f"3. **For inference**: Use {optimal_config['recommended_for_gui']['config']} with {optimal_config['recommended_for_gui']['blend_ratio']:.0%} blend ratio\n")
        f.write("4. **For training**: Focus on cross-attention and mid-block components\n\n")

        f.write("## Files Generated\n\n")
        f.write("- `component_importance.png`: Visual component ranking\n")
        f.write("- `quantitative_analysis.png`: Detailed metrics visualization\n")
        f.write("- `gallery/`: Interactive audio comparison gallery\n")
        f.write("- `detailed_metrics.json`: Complete quantitative results\n\n")

    # Return the path to the generated report
    return report_path

### 6.6 Execution Function for Full Ablation Study

In [18]:
def run_complete_ablation_study(model_path, output_dir=None):
    """Execute ablation study with focus on single instrument quality"""

    if output_dir is None:
        output_dir = os.path.join(OUTPUT_PATH, f"complete_ablation_{time.strftime('%Y%m%d_%H%M%S')}")

    os.makedirs(output_dir, exist_ok=True)

    print("=== Starting Ablation Study for Single Instrument Generation ===")

    # 1. Component-wise ablation
    print("\n1. Running component-wise ablation...")
    component_results = perform_component_ablation(model_path, output_dir)

    # 2. Blend ratio analysis
    print("\n2. Analyzing blend ratios...")
    blend_results = {}
    test_prompts = [
        "single drum hit, clean kick",
        "isolated piano key, single note",
        "clean synth pad, one chord",
        "vocal hit, single word",
        "guitar string pluck, one note"
    ]

    for ratio in [0.0, 0.25, 0.5, 0.75, 1.0]:
        print(f"  Testing blend ratio: {ratio}")

        # Generate samples at different ratios
        combined_score = 0
        for prompt in test_prompts:
            spec_img, audio_bytes, duration, seed = generate_with_blend_ratio(
                prompt=prompt,
                model_path=model_path,
                blend_ratio=ratio,
                seed=42
            )

            # Evaluate single instrument quality
            audio_data = np.frombuffer(audio_bytes.getvalue(), dtype=np.int16).astype(float) / 32768.0
            metrics = evaluate_single_source(prompt, audio_data=audio_data)

            # Calculate composite score
            score = np.mean([v for v in metrics.values() if isinstance(v, (int, float))])
            combined_score += score

        blend_results[ratio] = {
            "audio_quality": combined_score / len(test_prompts),
            "single_instrument_score": combined_score / len(test_prompts)
        }

    # 3. Find optimal configuration
    print("\n3. Determining optimal configuration...")
    optimal_config = find_optimal_model_configuration(
        component_results,
        blend_results,
        criteria='single_instrument'
    )

    # 4. Create audio gallery with optimal settings
    print("\n4. Creating audio gallery with optimal configuration...")
    gallery_widget, generated_samples, html_gallery = create_audio_gallery(
        model_path,
        os.path.join(output_dir, "gallery"),
        save_audio=True
    )

    # 5. Generate final report with recommendations
    print("\n5. Generating final report...")
    final_report = generate_final_report(component_results, blend_results, output_dir, optimal_config)

    print(f"\n=== Ablation Study Complete ===")
    print(f"Results saved to: {output_dir}")
    print(f"\nOptimal Configuration for GUI:")
    print(f"  Component Config: {optimal_config['optimal_component_config']}")
    print(f"  Blend Ratio: {optimal_config['optimal_blend_ratio']:.2f}")
    print(f"  GUI Settings: {optimal_config['recommended_for_gui']}")

    return {
        "component_results": component_results,
        "blend_results": blend_results,
        "optimal_config": optimal_config,
        "gallery_widget": gallery_widget,
        "output_directory": output_dir
    }

### 6.7 Execute complete study

In [ ]:
# Execute the complete ablation study
print("Starting comprehensive ablation study...")

# Set model path
model_path = "/content/drive/MyDrive/riffusion_finetuning/optimized_20250509-033329/best_model"

# Run complete study
results = run_complete_ablation_study(model_path)

# Display the gallery widget
print("\nDisplaying Audio Gallery:")
display(results["gallery_widget"])

## 7. User Interface

To allow user interaction, this section integrates the generation pipeline into a Gradio web interface for real-time prompt-to-audio conversion with the fine-tuned model.

### 7.1 Needed GUI Dependencies

In [64]:
import gradio as gr
import base64
from datetime import datetime

### 7.2 Model Loading and Configuration

In [65]:
# Load optimized model
BEST_MODEL_PATH = "/content/drive/MyDrive/riffusion_finetuning/optimized_20250509-033329/best_model"

# Initialize global variables for model management
global_pipeline = None
generation_history = []

def load_model():
    """Load the fine-tuned model with optimal settings - simplified approach"""
    global global_pipeline

    if global_pipeline is None:
        print("Loading model...")
        try:
            device = "cuda" if torch.cuda.is_available() else "cpu"


            # Load as regular PyTorch model
            global_pipeline = DiffusionPipeline.from_pretrained(
                MODEL_ID,
                torch_dtype=torch.float32,
                safety_checker=None,
                requires_safety_checker=False
            ).to(device)

            # Load fine-tuned weights
            unet_path = os.path.join(BEST_MODEL_PATH, "unet.pt")
            if os.path.exists(unet_path):
                global_pipeline.unet.load_state_dict(torch.load(unet_path, map_location=device))
                print(f"Loaded fine-tuned weights from {unet_path}")
            else:
                print(f"Warning: Could not find fine-tuned weights at {unet_path}")

            print("Model loaded successfully!")
            return True

        except Exception as e:
            print(f"Error loading model: {e}")
            traceback.print_exc()
            return False

    return True

def unload_model():
    """Manually unload model to free GPU memory"""
    global global_pipeline

    if global_pipeline is not None:
        del global_pipeline
        global_pipeline = None

        # Force garbage collection
        gc.collect()
        torch.cuda.empty_cache() if torch.cuda.is_available() else None

        return "Model unloaded successfully. GPU memory freed."
    else:
        return "Model was not loaded."

### 7.3 Prompt Enhancement Setup

In [66]:
# Prompt enhancement templates
GENRE_OPTIONS = ["EDM", "Dubstep", "Trap", "Techno", "House", "Trance", "Drum & Bass", "Ambient"]
INSTRUMENT_OPTIONS = ["Drums", "Bass", "Synth", "Piano", "Guitar", "Vocals", "Percussion", "Strings"]
BPM_RANGES = {"Slow (60-90)": 75, "Medium (90-130)": 110, "Fast (130-180)": 150, "Very Fast (180+)": 190}

def enhance_prompt(base_prompt: str, genre: str = None, instrument: str = None,
                  bpm: str = None, additional_options: List[str] = None) -> str:
    """Enhance user prompt with selected options"""

    enhanced_parts = [base_prompt]

    # Add genre
    if genre and genre != "None":
        enhanced_parts.append(f"{genre.lower()} style")

    # Add instrument type
    if instrument and instrument != "None":
        enhanced_parts.append(f"{instrument.lower()} sound")

    # Add BPM
    if bpm and bpm != "None":
        if bpm in BPM_RANGES:
            actual_bpm = BPM_RANGES[bpm]
            enhanced_parts.append(f"{actual_bpm} BPM")
        else:
            enhanced_parts.append(f"{bpm}")

    # Add additional options
    if additional_options:
        for option in additional_options:
            if option.strip():
                enhanced_parts.append(option.strip())

    # Clean and join parts
    enhanced_prompt = ", ".join(part for part in enhanced_parts if part.strip())

    # Add quality descriptors
    enhanced_prompt += ", professional quality, clean audio"

    return enhanced_prompt


def generate_with_style_transfer(
    prompt: str,
    reference_image: Image.Image,
    edge_region: Image.Image,
    model_path: str,
    blend_ratio: float = 0.75,
    guidance_scale: float = 8.0,
    steps: int = 25,
    denoising_strength: float = 0.75,
    seed: int = None
) -> Tuple[Optional[Image.Image], Optional[bytes], Optional[float], int]:
    """
    Generate using style transfer from previous segment
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load models for img2img
    pipeline = DiffusionPipeline.from_pretrained(
        "riffusion/riffusion-model-v1",
        torch_dtype=torch.float32,
        safety_checker=None
    ).to(device)

    # Load fine-tuned weights
    if os.path.isdir(model_path):
        unet_path = os.path.join(model_path, "unet.pt")
    else:
        unet_path = model_path

    if os.path.exists(unet_path):
        pipeline.unet.load_state_dict(torch.load(unet_path, map_location=device))

    # Set seed
    if seed is None:
        seed = random.randint(0, 2**32-1)

    generator = torch.Generator(device=device).manual_seed(seed)

    # Use img2img with the edge region
    result = pipeline(
        prompt=prompt,
        image=edge_region,
        strength=denoising_strength,  # Lower = more like original
        guidance_scale=guidance_scale,
        num_inference_steps=steps,
        generator=generator
    )

    # Convert to audio
    audio_bytes, duration = wav_bytes_from_spectrogram_image(result.images[0])

    return result.images[0], audio_bytes, duration, seed


def extract_metadata(prompt: str) -> Dict[str, str]:
    """Extract metadata from user prompt for history tracking"""

    metadata = {
        "genre": "Unknown",
        "instrument": "Unknown",
        "bpm": "Unknown",
        "style": "Unknown"
    }

    prompt_lower = prompt.lower()

    # Extract genre
    for genre in GENRE_OPTIONS:
        if genre.lower() in prompt_lower:
            metadata["genre"] = genre
            break

    # Extract instrument
    for instrument in INSTRUMENT_OPTIONS:
        if instrument.lower() in prompt_lower:
            metadata["instrument"] = instrument
            break

    # Extract BPM
    import re
    bpm_match = re.search(r'(\d+)\s*bpm', prompt_lower)
    if bpm_match:
        metadata["bpm"] = bpm_match.group(1)

    return metadata

### 7.4 Core Generation Function with Outpainting

In [67]:
def generate_audio_sample(
    prompt: str,
    negative_prompt: str = "",
    num_inference_steps: int = 25,
    guidance_scale: float = 8.0,
    seed: int = None,
    sample_length: int = 5,
    blend_ratio: float = 0.75,
    enable_outpainting: bool = False,
    num_segments: int = 1
) -> Tuple[Optional[Image.Image], Optional[bytes], Optional[float], Dict]:
    """
    Generate audio with style transfer for outpainting
    """

    if not load_model():
        return None, None, None, {"error": "Failed to load model"}

    try:
        # Set seed
        if seed is None:
            seed = random.randint(0, 2**32-1)

        # Calculate segments (5 seconds each)
        if enable_outpainting and sample_length > 5:
            segments_needed = (sample_length + 4) // 5  # Round up
            segments_needed = min(segments_needed, 3)  # Cap at 3
        else:
            segments_needed = 1

        audio_segments = []
        spectrograms = []

        # Generate first segment normally
        spec_img, audio_bytes, duration, _ = generate_with_blend_ratio(
            prompt=prompt,
            model_path=BEST_MODEL_PATH,
            blend_ratio=blend_ratio,
            blend_type="latent",
            guidance_scale=guidance_scale,
            steps=num_inference_steps,
            seed=seed
        )

        if spec_img and audio_bytes:
            spectrograms.append(spec_img)
            audio_bytes.seek(0)
            audio_segments.append(pydub.AudioSegment.from_file(io.BytesIO(audio_bytes.read())))

        # Generate subsequent segments using style transfer
        for i in range(1, segments_needed):
            if not spectrograms:
                break

            # Extract edge region from previous segment
            prev_spec = spectrograms[-1]
            edge_width = 50  # Pixels to use for style reference
            edge_region = prev_spec.crop((prev_spec.width - edge_width, 0, prev_spec.width, prev_spec.height))

            # Create a full-width init image by repeating the edge
            init_image = Image.new('RGB', (512, 512))
            for x in range(0, 512, edge_width):
                init_image.paste(edge_region, (x, 0))

            # Generate with style transfer
            segment_seed = seed + (i * 11)

            # Modify prompt for style consistency
            style_prompt = f"{prompt}, continuation, same style as previous section"

            spec_img, audio_bytes, duration, _ = generate_with_style_transfer(
                prompt=style_prompt,
                reference_image=prev_spec,
                edge_region=init_image,
                model_path=BEST_MODEL_PATH,
                blend_ratio=blend_ratio,
                guidance_scale=guidance_scale * 0.9,  # Slightly lower for style consistency
                steps=num_inference_steps,
                denoising_strength=0.65,  # Key parameter: lower = more like previous style
                seed=segment_seed
            )

            if spec_img and audio_bytes:
                spectrograms.append(spec_img)
                audio_bytes.seek(0)
                audio_segments.append(pydub.AudioSegment.from_file(io.BytesIO(audio_bytes.read())))

        # Combine segments with smooth transitions
        if len(audio_segments) > 1:
            # Create combined spectrogram with overlap blending
            overlap_width = 32
            total_width = sum(spec.width for spec in spectrograms) - (overlap_width * (len(spectrograms) - 1))
            combined_spec = Image.new('RGB', (total_width, spectrograms[0].height))

            # Blend spectrograms
            x_offset = 0
            for i, spec in enumerate(spectrograms):
                if i == 0:
                    combined_spec.paste(spec, (0, 0))
                    x_offset = spec.width - overlap_width
                else:
                    # Create gradient mask for smooth blending
                    mask = Image.new('L', (spec.width, spec.height), 255)
                    for x in range(overlap_width):
                        alpha = x / overlap_width
                        mask.paste(int(255 * alpha), (x, 0, x+1, spec.height))

                    # Blend the images
                    spec_rgba = spec.copy()
                    spec_rgba.putalpha(mask)
                    overlay_area = combined_spec.crop((x_offset, 0, x_offset + spec.width, spec.height))
                    overlay_rgba = overlay_area.copy()
                    overlay_rgba.putalpha(255)
                    blended = Image.alpha_composite(overlay_rgba.convert('RGBA'), spec_rgba.convert('RGBA'))
                    combined_spec.paste(blended.convert('RGB'), (x_offset, 0))
                    x_offset += spec.width - overlap_width

            # Combine audio
            combined_audio = audio_segments[0]
            for segment in audio_segments[1:]:
                # Calculate crossfade duration
                segment_duration_ms = len(segment)
                crossfade_ms = min(500, int((overlap_width / spectrograms[0].width) * segment_duration_ms))
                combined_audio = combined_audio.append(segment, crossfade=crossfade_ms)

            # Trim to requested length
            if sample_length < 30:
                combined_audio = combined_audio[:sample_length * 1000]

            # Export final audio
            final_audio_bytes = io.BytesIO()
            combined_audio.export(final_audio_bytes, format="wav")
            final_audio_bytes.seek(0)
            final_duration = combined_audio.duration_seconds

        else:
            combined_spec = spectrograms[0] if spectrograms else None
            combined_audio = audio_segments[0] if audio_segments else None
            final_audio_bytes = io.BytesIO()
            if combined_audio:
                combined_audio.export(final_audio_bytes, format="wav")
                final_audio_bytes.seek(0)
                final_duration = combined_audio.duration_seconds
            else:
                final_duration = None

        # Create metadata
        generation_info = {
            "prompt": prompt,
            "negative_prompt": negative_prompt,
            "inference_steps": num_inference_steps,
            "guidance_scale": guidance_scale,
            "seed": seed,
            "blend_ratio": blend_ratio,
            "segments": segments_needed,
            "duration": final_duration,
            "timestamp": datetime.now().isoformat(),
            "method": "style_transfer_outpainting"
        }

        # Add to history
        if combined_spec and final_audio_bytes:
            generation_history.append({
                "image": combined_spec,
                "audio": final_audio_bytes,
                "info": generation_info,
                "metadata": extract_metadata(prompt)
            })

        return combined_spec, final_audio_bytes, final_duration, generation_info

    except Exception as e:
        error_msg = f"Generation error: {str(e)}\n{traceback.format_exc()}"
        print(error_msg)
        return None, None, None, {"error": error_msg}

### 7.5 History Management System

In [68]:
def get_generation_history() -> List[Dict]:
    """Get formatted generation history"""

    history_display = []
    for i, item in enumerate(reversed(generation_history)):
        info = item["info"]
        metadata = item["metadata"]

        history_entry = {
            "index": len(generation_history) - i,
            "prompt": info["prompt"][:50] + "..." if len(info["prompt"]) > 50 else info["prompt"],
            "genre": metadata.get("genre", "Unknown"),
            "instrument": metadata.get("instrument", "Unknown"),
            "duration": f"{info.get('duration', 0):.1f}s",
            "timestamp": info["timestamp"].split('T')[1].split('.')[0],  # Time only
            "full_item": item
        }
        history_display.append(history_entry)

    return history_display

def save_generation(index: int) -> Tuple[str, str]:
    """Save a specific generation from history"""

    if 0 <= index < len(generation_history):
        item = generation_history[-(index + 1)]  # Reverse index for chronological order

        # Save audio
        audio_filename = f"riffusion_gen_{index}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.wav"
        audio_path = f"/tmp/{audio_filename}"

        item["audio"].seek(0)
        with open(audio_path, 'wb') as f:
            f.write(item["audio"].read())

        # Save spectrogram
        img_filename = f"riffusion_spec_{index}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        img_path = f"/tmp/{img_filename}"

        item["image"].save(img_path)

        return audio_path, img_path

    return None, None

def clear_history():
    """Clear generation history"""
    global generation_history
    generation_history = []
    return "History cleared"

### 7.6 Gradio Interface components

In [69]:
def create_interface():
    """Create the main Gradio interface"""

    with gr.Blocks(title="Riffusion Fine-tuned Sample Generator", theme=gr.themes.Soft()) as interface:

        gr.Markdown("# 🎵 Riffusion Sample Generator")
        gr.Markdown("Generate professional music production samples using the fine-tuned Riffusion model")

        with gr.Row():
            with gr.Column(scale=3):
                # Basic Input Section
                prompt_input = gr.Textbox(
                    label="Main Prompt",
                    placeholder="Enter your prompt (e.g., 'heavy dubstep bass drop')",
                    lines=2
                )

                # Quick Options
                with gr.Row():
                    genre_dropdown = gr.Dropdown(
                        choices=["None"] + GENRE_OPTIONS,
                        value="None",
                        label="Genre"
                    )
                    instrument_dropdown = gr.Dropdown(
                        choices=["None"] + INSTRUMENT_OPTIONS,
                        value="None",
                        label="Instrument Type"
                    )

                with gr.Row():
                    bpm_dropdown = gr.Dropdown(
                        choices=["None"] + list(BPM_RANGES.keys()),
                        value="None",
                        label="Tempo"
                    )
                    sample_length = gr.Slider(
                        minimum=5,
                        maximum=30,
                        value=5,
                        step=5,
                        label="Sample Length (seconds)"
                    )

                # Advanced Options (Accordion)
                with gr.Accordion("Advanced Options", open=False):
                    negative_prompt = gr.Textbox(
                        label="Negative Prompt",
                        placeholder="What to avoid (e.g., 'distorted, low quality')",
                        lines=1
                    )

                    with gr.Row():
                        inference_steps = gr.Slider(
                            minimum=10,
                            maximum=50,
                            value=25,
                            step=5,
                            label="Inference Steps"
                        )
                        guidance_scale = gr.Slider(
                            minimum=1.0,
                            maximum=20.0,
                            value=8.0,
                            step=0.5,
                            label="Guidance Scale"
                        )

                    with gr.Row():
                        blend_ratio = gr.Slider(
                            minimum=0.0,
                            maximum=1.0,
                            value=1.00,
                            step=0.05,
                            label="Fine-tuned/Baseline Blend (1 For the Full Fine Tuned Model)"
                        )
                        seed_input = gr.Number(
                            label="Seed (optional)",
                            precision=0
                        )

                    enable_outpainting = gr.Checkbox(
                        label="Enable Outpainting (for longer samples)",
                        value=True
                    )

                # Generation Controls
                with gr.Row():
                    generate_btn = gr.Button("🎵 Generate", variant="primary", scale=2)
                    enhance_prompt_btn = gr.Button("✨ Enhance Prompt", scale=1)

                # Enhanced Prompt Display
                enhanced_prompt_display = gr.Textbox(
                    label="Enhanced Prompt",
                    lines=2,
                    interactive=False
                )

                # Model Management
                with gr.Row():
                    unload_btn = gr.Button("🔄 Unload Model (Free GPU Memory)")
                    model_status = gr.Textbox(
                        label="Model Status",
                        value="Model not loaded",
                        interactive=False
                    )

            with gr.Column(scale=2):
                # Output Section
                spectrogram_output = gr.Image(
                    label="Generated Spectrogram",
                    type="pil"
                )

                audio_output = gr.Audio(
                    label="Generated Audio",
                    type="filepath"
                )

                generation_info = gr.JSON(
                    label="Generation Info",
                    visible=True
                )

        # History Section
        gr.Markdown("## 📚 Generation History")

        with gr.Row():
            with gr.Column():
                refresh_history_btn = gr.Button("🔄 Refresh History")
                clear_history_btn = gr.Button("🗑️ Clear History")

        history_df = gr.Dataframe(
            headers=["#", "Prompt", "Genre", "Instrument", "Duration", "Time"],
            datatype=["number", "str", "str", "str", "str", "str"],
            interactive=False,
            wrap=True
        )

        with gr.Row():
            history_index = gr.Number(
                label="History Item #",
                precision=0,
                minimum=0
            )
            save_history_btn = gr.Button("💾 Save Selected Item")
            download_audio = gr.File(label="Download Audio")
            download_image = gr.File(label="Download Spectrogram")

        # Event Handlers
        def update_model_status():
            if global_pipeline is not None:
                return "Model loaded and ready"
            else:
                if load_model():
                    return "Model loaded and ready"
                else:
                    return "Model failed to load"

        def enhance_prompt_handler(prompt, genre, instrument, bpm):
            enhanced = enhance_prompt(prompt, genre, instrument, bpm)
            return enhanced

        def generate_handler(prompt, negative_prompt, steps, guidance, seed, length, blend, enable_outpaint, *args):
            if not prompt.strip():
                return None, None, {"error": "Please enter a prompt"}, "No prompt provided"

            status = update_model_status()
            if "failed" in status:
                return None, None, {"error": "Model failed to load"}, status

            # Determine segments for outpainting
            num_segments = 1
            if enable_outpaint and length > 5:
                num_segments = min(3, (length // 5) + (1 if length % 5 > 0 else 0))

            img, audio, duration, info = generate_audio_sample(
                prompt=prompt,
                negative_prompt=negative_prompt,
                num_inference_steps=int(steps),
                guidance_scale=guidance,
                seed=int(seed) if seed else None,
                sample_length=int(length),
                blend_ratio=blend,
                enable_outpainting=enable_outpaint,
                num_segments=num_segments
            )

            if img is None or audio is None:
                error_info = info.get("error", "Unknown error occurred")
                return None, None, {"error": error_info}, f"Generation failed: {error_info}"

            # Save audio to temporary file for Gradio
            temp_audio_path = f"/tmp/generated_audio_{datetime.now().strftime('%Y%m%d_%H%M%S')}.wav"
            audio.seek(0)
            with open(temp_audio_path, 'wb') as f:
                f.write(audio.read())

            return img, temp_audio_path, info, "Generation complete!"

        def refresh_history_handler():
            history = get_generation_history()
            if not history:
                return []

            # Format for dataframe
            df_data = []
            for item in history:
                df_data.append([
                    item["index"],
                    item["prompt"],
                    item["genre"],
                    item["instrument"],
                    item["duration"],
                    item["timestamp"]
                ])

            return df_data

        def save_history_handler(index):
            if index is None:
                return None, None

            audio_path, img_path = save_generation(int(index))
            return audio_path, img_path

        # Connect event handlers
        generate_btn.click(
            fn=generate_handler,
            inputs=[
                prompt_input, negative_prompt, inference_steps, guidance_scale,
                seed_input, sample_length, blend_ratio, enable_outpainting
            ],
            outputs=[spectrogram_output, audio_output, generation_info, model_status]
        )

        enhance_prompt_btn.click(
            fn=enhance_prompt_handler,
            inputs=[prompt_input, genre_dropdown, instrument_dropdown, bpm_dropdown],
            outputs=[enhanced_prompt_display]
        )

        unload_btn.click(
            fn=unload_model,
            outputs=[model_status]
        )

        refresh_history_btn.click(
            fn=refresh_history_handler,
            outputs=[history_df]
        )

        clear_history_btn.click(
            fn=clear_history,
            outputs=[model_status]
        )

        save_history_btn.click(
            fn=save_history_handler,
            inputs=[history_index],
            outputs=[download_audio, download_image]
        )

        # Load model on startup
        interface.load(
            fn=update_model_status,
            outputs=[model_status]
        )

    return interface

### 7.7 Launch Interface

In [ ]:
# Launch the interface
if __name__ == "__main__":
    # Create and launch the interface
    interface = create_interface()
    interface.launch(
        share=True,
        debug=True,
        server_name="0.0.0.0",
        server_port=7869
    )

In [ ]:
try:
    # Close all open Gradio interfaces
    gr.close_all()
    print("Closed existing interfaces")
except:
    pass
